# Unlearning Project — Full Pre-LangChain A/B Experiment

**Protocol:** `prelangchain_ab_v1`  
**Unit of analysis:** one target paragraph per request  
**Safe default:** no paid API calls  
**Few-shot examples:** excluded from this notebook; deferred to the LangChain phase

This notebook is a staged, auditable experiment for selecting the definition, prompt structure, context window, and classification workflow before introducing LangChain. It keeps OpenAI, Anthropic, and Gemini execution in separate cells and separate result files so one provider failure cannot damage another provider's completed work.

The central experiment compares:

- **D1 — old broad definition**;
- **D2 — current strict definition**;
- **D3 — provisional adaptive-reconfiguration definition**, designed to test whether the EPA failures arise partly from a definition that excludes durable post-failure changes in roles, protocols, coordination, and technical routines.

D3 is an exploratory research treatment, not a replacement for later human adjudication.

## Experimental sequence

| Phase | Research question | Conditions |
|---|---|---|
| 0 | Are the benchmark, context blocks, definitions, schemas, and provider SDKs valid? | integrity gates and smoke tests |
| 1 | Which definition and prompt structure work best? | P1 direct; P2 simple definition; P3 evidence checklist × D1/D2/D3 |
| 2 | How much context is useful? | C1 target; C2 metadata; C3 ±1; C4 ±2 on the two Phase-1 finalists |
| 3 | Is a hierarchical workflow better? | W1 one-stage joint vs W2 binary-first then target/agency |
| 4 | Are the finalists stable? | repeated runs with seeds 17, 43, 101, 211, 307 |
| 5 | How should production review work? | fixed three-provider tiered review policy |

Selection is lexicographic: schema/evidence validity and specificity constraints first, then worst-document recall, F1, MCC, balanced accuracy, stability, and finally cost/latency.

## Reproducibility principles

- Historical labels are never overwritten.
- Definition-aligned human gold columns are used when complete; otherwise results are explicitly marked provisional against the historical labels.
- Inputs, definitions, schemas, prompts, model configurations, requests, raw responses, and reports are hashed.
- A seed controls local scheduling, preprocessing, retry jitter, and provider seed fields only when verified. Temperature 0 and a seed do **not** guarantee deterministic hosted-model output.
- Every successful request is resumable by deterministic run key.
- Raw JSONL is append-only. Summary tables are regenerated from raw logs.
- Provider configuration errors stop only that provider's cell.
- A neighboring paragraph cannot independently make the target positive.
- No narrative rationale field is requested; only structured evidence and decisions are returned.

## 0. Optional dependency installation

In [ ]:
# Run once in a fresh environment, then restart the kernel.
#%pip install -U pandas numpy openpyxl pyarrow pydantic scikit-learn scipy statsmodels krippendorff python-dotenv tqdm nbformat openai anthropic google-genai

## 1. Imports

In [32]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, asdict, field, replace
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any, Callable, Iterable, Literal, Optional, Sequence, Type
import contextlib
import hashlib
import importlib.metadata
import inspect
import itertools
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
import traceback
import unicodedata
import uuid
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from pydantic import BaseModel, ConfigDict, Field
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score,
)

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.multitest import multipletests
except ImportError:
    sm = smf = multipletests = None

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 220)
warnings.filterwarnings('default')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Parameters

In [33]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [34]:
# PAPERMILL / NOTEBOOK PARAMETERS
PROTOCOL_VERSION = 'prelangchain_ab_v1'

PROJECT_ROOT = Path(os.getenv('UNLEARNING_PROJECT_ROOT', Path.cwd())).resolve()
INPUT_WORKBOOK = Path(os.getenv(
    'UNLEARNING_INPUT_WORKBOOK',
    PROJECT_ROOT / 'data' / 'Unlearning_Codebook_Local_Context_Test_Set.xlsx',
))
INPUT_SHEET = os.getenv('UNLEARNING_INPUT_SHEET', 'GPT Test')

RUNS_ROOT = PROJECT_ROOT / 'runs' / PROTOCOL_VERSION
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts' / PROTOCOL_VERSION
REPORTS_ROOT = PROJECT_ROOT / 'reports' / PROTOCOL_VERSION
CONFIG_ROOT = PROJECT_ROOT / 'configs' / PROTOCOL_VERSION

RUN_API_CALLS = os.getenv('UNLEARNING_RUN_API_CALLS', 'true').lower() == 'true'
API_RUN_CONFIRMATION = os.getenv('UNLEARNING_API_CONFIRMATION', 'RUN_PRELANGCHAIN_AB_V1')
REQUIRED_API_CONFIRMATION = 'RUN_PRELANGCHAIN_AB_V1'
USE_SYNTHETIC_DATA = os.getenv('UNLEARNING_USE_SYNTHETIC', 'false').lower() == 'true'
USE_MOCK_PROVIDER = os.getenv('UNLEARNING_USE_MOCK_PROVIDER', 'false').lower() == 'true'
ALLOW_PROVISIONAL_SELECTION = os.getenv(
    'UNLEARNING_ALLOW_PROVISIONAL_SELECTION', 'true'
).lower() == 'true'

RUN_REPLICATE_ID = os.getenv('UNLEARNING_REPLICATE_ID', 'r1')
ACTIVE_PHASES = tuple(x.strip() for x in os.getenv(
    'UNLEARNING_ACTIVE_PHASES', 'phase1,phase2,phase3,stability'
).split(',') if x.strip())
ACTIVE_SEEDS = tuple(int(x) for x in os.getenv('UNLEARNING_SEEDS', '17').split(',') if x.strip())
STABILITY_SEEDS = tuple(int(x) for x in os.getenv(
    'UNLEARNING_STABILITY_SEEDS', '17,43,101,211,307'
).split(',') if x.strip())

DISCOVERY_SEED = 17
MAX_OUTPUT_TOKENS = int(os.getenv('UNLEARNING_MAX_OUTPUT_TOKENS', '1800'))
MAX_STAGE2_OUTPUT_TOKENS = int(os.getenv('UNLEARNING_MAX_STAGE2_OUTPUT_TOKENS', '800'))
MAX_NEW_CALLS_PER_PROVIDER_CELL = int(os.getenv('UNLEARNING_MAX_NEW_CALLS_PER_PROVIDER_CELL', '5000'))
MAX_RETRY_ATTEMPTS = int(os.getenv('UNLEARNING_MAX_RETRY_ATTEMPTS', '4'))
BASE_RETRY_SECONDS = float(os.getenv('UNLEARNING_BASE_RETRY_SECONDS', '1.5'))
REQUEST_SPACING_SECONDS = float(os.getenv('UNLEARNING_REQUEST_SPACING_SECONDS', '0.1'))
STOP_PROVIDER_ON_CONFIGURATION_ERROR = True
RAISE_PROVIDER_CELL_EXCEPTIONS = False
PASS_PROVIDER_NATIVE_SEED = False

EXPECTED_BENCHMARK_ROWS = 42
EXPECTED_GOLD_YES = 32
EXPECTED_GOLD_NO = 10
TARGET_CONTEXT_MATCH_THRESHOLD = 0.97
NEAR_DUPLICATE_JACCARD_THRESHOLD = 0.90
SPECIFICITY_FLOOR = float(os.getenv('UNLEARNING_SPECIFICITY_FLOOR', '0.80'))
MIN_SCHEMA_VALID_RATE = float(os.getenv('UNLEARNING_MIN_SCHEMA_VALID_RATE', '0.99'))
MIN_EVIDENCE_VALID_RATE = float(os.getenv('UNLEARNING_MIN_EVIDENCE_VALID_RATE', '0.98'))
N_BOOTSTRAP = int(os.getenv('UNLEARNING_N_BOOTSTRAP', '2000'))
RUN_GEE_MODELS = os.getenv('UNLEARNING_RUN_GEE', 'true').lower() == 'true'

PHASE1_SELECTION_OVERRIDE = tuple(x.strip() for x in os.getenv(
    'UNLEARNING_PHASE1_SELECTION_OVERRIDE', ''
).split(',') if x.strip())
PHASE2_SELECTION_OVERRIDE = os.getenv('UNLEARNING_PHASE2_SELECTION_OVERRIDE', '').strip()
PHASE3_WORKFLOW_OVERRIDE = os.getenv('UNLEARNING_PHASE3_WORKFLOW_OVERRIDE', '').strip()

In [35]:
for path in [RUNS_ROOT, ARTIFACTS_ROOT, REPORTS_ROOT, CONFIG_ROOT, INPUT_WORKBOOK.parent]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG_SUMMARY = {
    'protocol_version': PROTOCOL_VERSION,
    'project_root': str(PROJECT_ROOT),
    'input_workbook': str(INPUT_WORKBOOK),
    'input_sheet': INPUT_SHEET,
    'run_api_calls': RUN_API_CALLS,
    'use_synthetic_data': USE_SYNTHETIC_DATA,
    'use_mock_provider': USE_MOCK_PROVIDER,
    'active_phases': ACTIVE_PHASES,
    'active_seeds': ACTIVE_SEEDS,
    'stability_seeds': STABILITY_SEEDS,
    'replicate_id': RUN_REPLICATE_ID,
}
display(pd.DataFrame([CONFIG_SUMMARY]).T.rename(columns={0: 'value'}))

,value
protocol_version,prelangchain_ab_v1
project_root,/content
input_workbook,/content/data/Unlearning_Codebook_Local_Context_Test_Set.xlsx
input_sheet,GPT Test
run_api_calls,True
use_synthetic_data,False
use_mock_provider,False
active_phases,"(phase1, phase2, phase3, stability)"
active_seeds,"(17,)"
stability_seeds,"(17, 43, 101, 211, 307)"


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. Deterministic local state, hashing, and environment capture

In [36]:
def set_global_seed(seed: int) -> dict[str, Any]:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    status = {'python': seed, 'numpy': seed, 'torch': False, 'tensorflow': False}
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        with contextlib.suppress(Exception):
            torch.use_deterministic_algorithms(True)
        status['torch'] = True
    except ImportError:
        pass
    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
        status['tensorflow'] = True
    except ImportError:
        pass
    return status

SEED_STATUS = set_global_seed(DISCOVERY_SEED)

def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'), default=str)

def sha256_text(value: Any) -> str:
    return hashlib.sha256(str(value).encode('utf-8')).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> Optional[str]:
    if not path.exists() or not path.is_file():
        return None
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def stable_int_seed(*parts: Any) -> int:
    digest = hashlib.sha256(canonical_json(parts).encode('utf-8')).hexdigest()
    return int(digest[:16], 16) % (2**32 - 1)

def package_version(name: str) -> Optional[str]:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

TRACKED_PACKAGES = [
    'pandas', 'numpy', 'openpyxl', 'pyarrow', 'pydantic', 'scikit-learn',
    'scipy', 'statsmodels', 'krippendorff', 'openai', 'anthropic',
    'google-genai', 'python-dotenv', 'tqdm', 'nbformat',
]
ENVIRONMENT_SNAPSHOT = {
    'captured_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': sys.version,
    'executable': sys.executable,
    'platform': platform.platform(),
    'seed_status': SEED_STATUS,
    'packages': {name: package_version(name) for name in TRACKED_PACKAGES},
}
(CONFIG_ROOT / 'environment_snapshot.json').write_text(
    json.dumps(ENVIRONMENT_SNAPSHOT, indent=2, default=str), encoding='utf-8'
)
display(pd.DataFrame([
    {'package': key, 'version': value}
    for key, value in ENVIRONMENT_SNAPSHOT['packages'].items()
]))

,package,version
0,pandas,3.0.3
1,numpy,2.5.1
2,openpyxl,3.1.5
3,pyarrow,25.0.0
4,pydantic,2.13.4
5,scikit-learn,1.9.0
6,scipy,1.18.0
7,statsmodels,0.14.6
8,krippendorff,0.8.2
9,openai,2.46.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### API execution guard

In [37]:
def assert_api_execution_allowed(provider: str) -> None:
    if USE_MOCK_PROVIDER:
        return
    if not RUN_API_CALLS:
        raise RuntimeError(f'{provider}: paid API calls are disabled.')
    if API_RUN_CONFIRMATION != REQUIRED_API_CONFIRMATION:
        raise RuntimeError(
            f'{provider}: set UNLEARNING_API_CONFIRMATION='
            f'{REQUIRED_API_CONFIRMATION!r} after reviewing the smoke test.'
        )
    env_name = {
        'openai': 'OPENAI_API_KEY',
        'anthropic': 'ANTHROPIC_API_KEY',
        'gemini': 'GEMINI_API_KEY or GOOGLE_API_KEY',
    }[provider]
    if provider == 'gemini':
        present = bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'))
    else:
        present = bool(os.getenv(env_name))
    if not present:
        raise RuntimeError(f'{provider}: missing {env_name}.')

print({
    'RUN_API_CALLS': RUN_API_CALLS,
    'confirmation_valid': API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION,
    'mock_provider': USE_MOCK_PROVIDER,
})

{'RUN_API_CALLS': True, 'confirmation_valid': True, 'mock_provider': False}


## 4. Fixed provider/model protocol

In [38]:
@dataclass(frozen=True)
class ProviderModelConfig:
    provider: Literal['openai', 'anthropic', 'gemini']
    model_id: str
    temperature: float
    reasoning_label: str
    reasoning_payload: Optional[dict[str, Any]]
    api_key_env: str
    prompt_caching: str
    max_output_tokens: int = MAX_OUTPUT_TOKENS

MODEL_CONFIGS: dict[str, ProviderModelConfig] = {
    'openai': ProviderModelConfig(
        provider='openai', model_id='gpt-5.6-terra', temperature=0.0,
        reasoning_label='reasoning.effort=low', reasoning_payload={'effort': 'low'},
        api_key_env='OPENAI_API_KEY',
        prompt_caching='prompt_cache_key with static prefix',
    ),
    'anthropic': ProviderModelConfig(
        provider='anthropic', model_id='claude-haiku-4-5-20251001', temperature=0.0,
        reasoning_label='lowest setting: extended thinking disabled', reasoning_payload=None,
        api_key_env='ANTHROPIC_API_KEY',
        prompt_caching='top-level cache_control=ephemeral',
    ),
    'gemini': ProviderModelConfig(
        provider='gemini', model_id='gemini-3.1-flash-lite', temperature=0.0,
        reasoning_label='thinking_level=low', reasoning_payload={'thinking_level': 'low'},
        api_key_env='GEMINI_API_KEY or GOOGLE_API_KEY',
        prompt_caching='implicit prefix caching; cached token usage logged',
    ),
}

def model_config_hash(config: ProviderModelConfig) -> str:
    return sha256_text(canonical_json(asdict(config)))

MODEL_PROTOCOL = pd.DataFrame([
    {**asdict(config), 'config_sha256': model_config_hash(config)}
    for config in MODEL_CONFIGS.values()
])
MODEL_PROTOCOL.to_csv(CONFIG_ROOT / 'model_protocol.csv', index=False)
display(MODEL_PROTOCOL)

,provider,model_id,temperature,reasoning_label,reasoning_payload,api_key_env,prompt_caching,max_output_tokens,config_sha256
0,openai,gpt-5.6-terra,0.0,reasoning.effort=low,{'effort': 'low'},OPENAI_API_KEY,prompt_cache_key with static prefix,1800,6a1b619da016ef6ef95e6b667fd1de7a452f2b5ac6c970c501dd3781c4e2a25a
1,anthropic,claude-haiku-4-5-20251001,0.0,lowest setting: extended thinking disabled,None,ANTHROPIC_API_KEY,top-level cache_control=ephemeral,1800,86832fd651e7d1d4efead0530bec98029d5d913a0d7f0252f5a429f982aeca91
2,gemini,gemini-3.1-flash-lite,0.0,thinking_level=low,{'thinking_level': 'low'},GEMINI_API_KEY or GOOGLE_API_KEY,implicit prefix caching; cached token usage logged,1800,512584f3fdc3a52c1b743d1b3314e18b6ccf9276fc94b38f3878d780010a47db


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Reasoning-setting parity

“Low reasoning” is provider-specific. OpenAI receives `reasoning.effort=low`; Gemini receives `thinking_level=low`; Claude Haiku 4.5 is run without extended thinking because the experiment also fixes temperature at 0. The accepted request parameters and returned model snapshot are logged on every call. No setting is silently dropped after a provider rejects it.

## 5. Registered definition treatments

In [39]:
@dataclass(frozen=True)
class DefinitionSpec:
    definition_id: str
    name: str
    status: str
    operational_text: str
    scientific_role: str

DEFINITIONS: dict[str, DefinitionSpec] = {
    'D0_none': DefinitionSpec(
        'D0_none', 'No supplied definition', 'control', '',
        'Direct-prompt baseline; ordinary model understanding.'
    ),
    'D1_old_broad': DefinitionSpec(
        'D1_old_broad', 'Old broad operational definition', 'historical treatment',
        """BROAD HISTORICAL OPERATIONALIZATION\n\nOrganizational unlearning is evidenced when a government organization reconsiders, discards, realigns, or merges established knowledge, assumptions, routines, roles, policies, capabilities, resources, or organizational arrangements after experience or failure.\n\nQualifying modes include epistemic reconsidering, normative discarding, technical realignment, and integrative merging. A passage may be positive even when it describes strategic integration, revised coordination, changed roles, technical adaptation, or additive institutional change rather than literal removal. Diagnosis alone is insufficient, but explicit words such as abandon or replace are not required.""",
        'Recover the broader construct used in early annotations.'
    ),
    'D2_current_strict': DefinitionSpec(
        'D2_current_strict', 'Current strict operational definition', 'current treatment',
        """CURRENT STRICT OPERATIONALIZATION\n\nOrganizational unlearning is present only when the passage identifies an earlier assumption, policy, practice, role arrangement, capability, resource logic, routine, or system as inadequate and indicates a meaningful departure from it through rejection, replacement, removal, abandonment, suspension, redistribution of authority, movement away from business as usual, or fundamental rethinking.\n\nDiagnosis, lessons-learned language, better implementation, extra training, extra staff, added equipment, more resources, and additive capacity are not sufficient unless the earlier approach being displaced is identifiable. Ordinary learning, reform, innovation, and improvement without discontinuity are No.""",
        'Create a sharp boundary between learning/reform and unlearning.'
    ),
    'D3_provisional_adaptive': DefinitionSpec(
        'D3_provisional_adaptive', 'Provisional adaptive-reconfiguration definition',
        'exploratory candidate; requires later human adjudication',
        """PROVISIONAL ADAPTIVE-RECONFIGURATION OPERATIONALIZATION\n\nOrganizational unlearning is present through either route.\n\nROUTE A — SUBTRACTIVE DISCONTINUITY: an identifiable earlier assumption, policy, practice, role arrangement, capability, resource logic, routine, or system is judged inadequate and is rejected, replaced, removed, abandoned, suspended, fundamentally rethought, or has authority redistributed.\n\nROUTE B — ADAPTIVE RECONFIGURATION: after a demonstrated failure, duplication, ambiguity, mismatch, or operational breakdown, the organization durably reconfigures formal roles, responsibility boundaries, coordination protocols, memoranda, standard operating procedures, decision rights, information routines, or technical operating routines for future events. The target must describe the institutionalized reconfiguration, not merely the problem.\n\nTraining, user guides, extra staff, equipment, laboratories, funding, or resources alone remain No. They count only when embedded in and subordinate to a qualifying change in the governing role, protocol, standard, authority structure, or routine. Temporary workarounds, implementation of an unchanged approach, diagnosis alone, and generic plans to improve remain No.""",
        'Test whether EPA recall can improve without treating all additive capacity as unlearning.'
    ),
}

DEFINITION_REGISTRY = pd.DataFrame([
    {**asdict(spec), 'sha256': sha256_text(spec.operational_text)}
    for spec in DEFINITIONS.values()
])
DEFINITION_REGISTRY.to_csv(CONFIG_ROOT / 'definition_registry.csv', index=False)
for spec in DEFINITIONS.values():
    (CONFIG_ROOT / f'{spec.definition_id}.txt').write_text(spec.operational_text, encoding='utf-8')
display(DEFINITION_REGISTRY[['definition_id','name','status','scientific_role','sha256']])

,definition_id,name,status,scientific_role,sha256
0,D0_none,No supplied definition,control,Direct-prompt baseline; ordinary model understanding.,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
1,D1_old_broad,Old broad operational definition,historical treatment,Recover the broader construct used in early annotations.,7531f2c94a782ca6a2642e01fca30d3bd631f94fb2a4e8fad4a90e82dfad513e
2,D2_current_strict,Current strict operational definition,current treatment,Create a sharp boundary between learning/reform and unlearning.,258f6556333fc5521c03d9aea3d43626ee22e7480b522f5e4c6729122e9e85ad
3,D3_provisional_adaptive,Provisional adaptive-reconfiguration definition,exploratory candidate; requires later human adjudication,Test whether EPA recall can improve without treating all additive capacity as unlearning.,99af0841b61738d090c0ecccff0f9c77d35f487944d743c52fa3dc11fcf595d5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### D3 interpretation guard

D3 is deliberately narrower than the original broad codebook. It can rescue cases involving formal liaison positions, changed responsibility boundaries, interagency protocols, revised SOPs, redistributed authority, or institutionalized technical routines after failure. It still excludes staff, training, equipment, laboratories, funding, and capacity additions by themselves. The notebook reports both EPA gains and non-EPA specificity costs.

## 6. Strict structured-output schemas — no narrative rationale

In [40]:
EvidenceLocation = Literal[
    'target', 'previous_1', 'previous_2', 'next_1', 'next_2', 'metadata', 'absent'
]
EvidenceElementName = Literal[
    'prior_state', 'inadequacy_or_failure', 'departure_or_reconfiguration', 'other'
]
TargetType = Literal[
    'leadership', 'laws_plans_policies', 'capabilities',
    'funds_resources', 'misc_organizational', 'none'
]
UnlearningMode = Literal[
    'subtractive_discontinuity', 'adaptive_reconfiguration',
    'epistemic_reconsidering', 'technical_realignment',
    'integrative_merging', 'none', 'unclear'
]
ChangeType = Literal[
    'reject', 'replace', 'remove', 'abandon', 'suspend',
    'fundamentally_rethink', 'redistribute_authority',
    'revise_protocol_or_standard', 'integrate_roles_or_plans',
    'add_capacity_only', 'diagnosis_only', 'no_change', 'unclear'
]

class StrictOutputModel(BaseModel):
    model_config = ConfigDict(extra='forbid')

class EvidenceQuote(StrictOutputModel):
    element: EvidenceElementName
    quote: str | None = Field(description='Exact short excerpt from supplied text, or null.')
    source_scope: EvidenceLocation

class ChecklistElement(StrictOutputModel):
    identified: bool
    quote: str | None = Field(description='Exact short excerpt from supplied text, or null.')
    source_scope: EvidenceLocation

class SimpleJointOutput(StrictOutputModel):
    unlearning_present: bool
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    evidence: list[EvidenceQuote]
    target_type: TargetType
    agency: str | None
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class ChecklistJointOutput(StrictOutputModel):
    prior_state: ChecklistElement
    inadequacy_or_failure: ChecklistElement
    departure_or_reconfiguration: ChecklistElement
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    all_required_elements_present: bool
    missing_elements: list[Literal[
        'prior_state', 'inadequacy_or_failure', 'departure_or_reconfiguration'
    ]]
    unlearning_present: bool
    target_type: TargetType
    agency: str | None
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class SimpleBinaryOutput(StrictOutputModel):
    unlearning_present: bool
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    evidence: list[EvidenceQuote]
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class ChecklistBinaryOutput(StrictOutputModel):
    prior_state: ChecklistElement
    inadequacy_or_failure: ChecklistElement
    departure_or_reconfiguration: ChecklistElement
    unlearning_mode: UnlearningMode
    change_type: ChangeType
    all_required_elements_present: bool
    missing_elements: list[Literal[
        'prior_state', 'inadequacy_or_failure', 'departure_or_reconfiguration'
    ]]
    unlearning_present: bool
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

class Stage2TargetOutput(StrictOutputModel):
    target_type: TargetType
    agency: str | None
    target_evidence_quote: str | None
    target_evidence_scope: EvidenceLocation
    confidence: float = Field(ge=0.0, le=1.0)
    needs_human_review: bool

SCHEMA_MODELS: dict[str, Type[BaseModel]] = {
    'simple_joint': SimpleJointOutput,
    'checklist_joint': ChecklistJointOutput,
    'simple_binary': SimpleBinaryOutput,
    'checklist_binary': ChecklistBinaryOutput,
    'stage2_target': Stage2TargetOutput,
}
SCHEMA_AUDIT = pd.DataFrame([
    {
        'schema_id': schema_id,
        'fields': ', '.join(model.model_fields),
        'has_rationale_field': 'rationale' in model.model_fields,
        'sha256': sha256_text(canonical_json(model.model_json_schema())),
    }
    for schema_id, model in SCHEMA_MODELS.items()
])
assert not SCHEMA_AUDIT['has_rationale_field'].any()
display(SCHEMA_AUDIT)

,schema_id,fields,has_rationale_field,sha256
0,simple_joint,"unlearning_present, unlearning_mode, change_type, evidence, target_type, agency, confidence, needs_human_review",False,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e
1,checklist_joint,"prior_state, inadequacy_or_failure, departure_or_reconfiguration, unlearning_mode, change_type, all_required_elements_present, missing_elements, unlearning_present, target_type, agency, confidence, needs_human_review",False,160ef51b848bbba94ee5996a63b20821792de7b7319a848da6741e11e321f8b6
2,simple_binary,"unlearning_present, unlearning_mode, change_type, evidence, confidence, needs_human_review",False,247094025bbf94277cb5a6a9a2c817fdfc99013e4f7cf6a1adae2f2ed4951137
3,checklist_binary,"prior_state, inadequacy_or_failure, departure_or_reconfiguration, unlearning_mode, change_type, all_required_elements_present, missing_elements, unlearning_present, confidence, needs_human_review",False,509b21ba3a52fffc80899c327afc9f42e73bc2ce91a695ea01463c6cb5ba17bc
4,stage2_target,"target_type, agency, target_evidence_quote, target_evidence_scope, confidence, needs_human_review",False,9a6c02f76d61ee3d74f8687ae16f157a02d9bd113cc3d7ec9e181ddb616c6509


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 7. Benchmark loading, context parsing, and freeze

In [41]:
COLUMN_ALIASES = {
    'Number': 'row_id', 'Row ID': 'row_id',
    'Text Content': 'target_text', 'Paragraph Text': 'target_text',
    'Document': 'document_title', 'Document Title': 'document_title',
    'PDF Page': 'pdf_page', 'Page(s)': 'pdf_page',
    'Section Heading': 'section_heading', 'Section/Location': 'section_heading',
    'Paragraph Order': 'paragraph_order',
    'Local Context (±2 source paragraphs)': 'context_raw',
    'Unlearning': 'gold_historical', 'Unlearning?': 'gold_historical',
    'Original Unlearning': 'gold_historical',
    'Target': 'gold_target', 'Original Target': 'gold_target',
    'Government Agency': 'gold_agency', 'Original Government Agency': 'gold_agency',
    'Gold Old Definition': 'gold_old_definition',
    'Gold Current Definition': 'gold_current_definition',
    'Gold Final Definition': 'gold_final_definition',
}
REQUIRED_CANONICAL_COLUMNS = {
    'row_id', 'target_text', 'document_title', 'section_heading',
    'context_raw', 'gold_historical'
}
OPTIONAL_GOLD_COLUMNS = [
    'gold_old_definition', 'gold_current_definition', 'gold_final_definition',
    'gold_target', 'gold_agency'
]

def synthetic_benchmark() -> pd.DataFrame:
    records = [
        dict(row_id='S001', target_text='The agency replaced the layered response model with a push model after the prior approach delayed assistance.', document_title='Synthetic GAO', pdf_page=1, section_heading='Response doctrine', paragraph_order=1, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nThe layered approach delayed aid.\n\n[TARGET]\nThe agency replaced the layered response model with a push model after the prior approach delayed assistance.\n\n[NEXT 1]\nThe new doctrine became standard.\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='laws_plans_policies', gold_agency='Synthetic agency'),
        dict(row_id='S002', target_text='The office added two mobile laboratories and additional staff.', document_title='Synthetic EPA', pdf_page=2, section_heading='Capacity', paragraph_order=2, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nAnalytical capacity was insufficient.\n\n[TARGET]\nThe office added two mobile laboratories and additional staff.\n\n[NEXT 1]\nMonitoring resumed.\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='capabilities', gold_agency='Synthetic EPA'),
        dict(row_id='S003', target_text='Region 6 established a formal liaison, defined agency responsibilities, and replaced informal coordination with an interagency protocol.', document_title='Synthetic EPA', pdf_page=3, section_heading='Coordination', paragraph_order=3, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nDuplicated work followed unclear roles.\n\n[TARGET]\nRegion 6 established a formal liaison, defined agency responsibilities, and replaced informal coordination with an interagency protocol.\n\n[NEXT 1]\nThe protocol governs future incidents.\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='misc_organizational', gold_agency='Region 6'),
        dict(row_id='S004', target_text='The report documented communication failures during the storm.', document_title='Synthetic EPA', pdf_page=4, section_heading='Findings', paragraph_order=4, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\n\n[TARGET]\nThe report documented communication failures during the storm.\n\n[NEXT 1]\n\n[NEXT 2]\n', gold_historical='No', gold_target='none', gold_agency=None),
        dict(row_id='S005', target_text='Officials provided refresher training on the existing procedure.', document_title='Synthetic Post-Katrina', pdf_page=5, section_heading='Training', paragraph_order=5, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nSome staff misunderstood the procedure.\n\n[TARGET]\nOfficials provided refresher training on the existing procedure.\n\n[NEXT 1]\nThe procedure itself was unchanged.\n\n[NEXT 2]\n', gold_historical='No', gold_target='none', gold_agency=None),
        dict(row_id='S006', target_text='The framework redistributed authority to local incident commanders and eliminated the former centralized approval requirement.', document_title='Synthetic Post-Katrina', pdf_page=6, section_heading='Authority', paragraph_order=6, context_raw='[PREVIOUS 2]\n\n[PREVIOUS 1]\nCentral approval caused delay.\n\n[TARGET]\nThe framework redistributed authority to local incident commanders and eliminated the former centralized approval requirement.\n\n[NEXT 1]\n\n[NEXT 2]\n', gold_historical='Yes', gold_target='leadership', gold_agency='Synthetic framework'),
    ]
    return pd.DataFrame(records)

In [42]:
def normalize_space(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

def normalize_for_match(value: Any) -> str:
    text = unicodedata.normalize('NFKC', normalize_space(value))
    translation = str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-','\u00a0':' '})
    return normalize_space(text.translate(translation)).casefold()

def normalize_label(value: Any) -> Optional[str]:
    text = normalize_space(value).casefold()
    if not text:
        return None
    if text in {'yes','y','true','1','unlearning','present'} or text.startswith('yes'):
        return 'Yes'
    if text in {'no','n','false','0','not unlearning','absent'} or text.startswith('no'):
        return 'No'
    return None

def label_to_int(value: Any) -> Optional[int]:
    return {'Yes':1, 'No':0}.get(normalize_label(value))

CONTEXT_MARKERS = ['PREVIOUS 2','PREVIOUS 1','TARGET','NEXT 1','NEXT 2']
CONTEXT_PATTERN = re.compile(r'\[(PREVIOUS 2|PREVIOUS 1|TARGET|NEXT 1|NEXT 2)\]\s*', flags=re.I)

def parse_context_blocks(raw: Any) -> dict[str, Any]:
    text = '' if raw is None else str(raw)
    matches = list(CONTEXT_PATTERN.finditer(text))
    blocks = {marker.lower().replace(' ','_'): '' for marker in CONTEXT_MARKERS}
    for index, match in enumerate(matches):
        start = match.end()
        stop = matches[index+1].start() if index+1 < len(matches) else len(text)
        blocks[match.group(1).lower().replace(' ','_')] = text[start:stop].strip()
    blocks['marker_count'] = len(matches)
    return blocks

def context_target_similarity(target: Any, parsed_target: Any) -> float:
    left, right = normalize_for_match(target), normalize_for_match(parsed_target)
    return SequenceMatcher(None, left, right).ratio() if (left or right) else 1.0

def canonicalize_benchmark(raw: pd.DataFrame) -> pd.DataFrame:
    data = raw.copy().dropna(axis=1, how='all')
    data = data.loc[:, ~data.columns.astype(str).str.match(r'^Unnamed')]
    data = data.rename(columns={k:v for k,v in COLUMN_ALIASES.items() if k in data.columns})
    missing = REQUIRED_CANONICAL_COLUMNS - set(data.columns)
    if missing:
        raise ValueError(f'Missing benchmark columns: {sorted(missing)}')
    for column in OPTIONAL_GOLD_COLUMNS:
        if column not in data.columns:
            data[column] = None
    for column in ['pdf_page','paragraph_order']:
        if column not in data.columns:
            data[column] = None
    data['row_id'] = data['row_id'].astype(str)
    for column in ['target_text','document_title','section_heading','context_raw']:
        data[column] = data[column].map(normalize_space)
    for column in ['gold_historical','gold_old_definition','gold_current_definition','gold_final_definition']:
        data[column] = data[column].map(normalize_label)
    parsed = data['context_raw'].map(parse_context_blocks).apply(pd.Series)
    parsed = parsed.rename(columns={
        'previous_2':'context_previous_2','previous_1':'context_previous_1',
        'target':'context_target','next_1':'context_next_1','next_2':'context_next_2'
    })
    data = pd.concat([data.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)
    data['context_target_similarity'] = data.apply(
        lambda row: context_target_similarity(row['target_text'], row['context_target']), axis=1
    )
    data['normalized_target_sha256'] = data['target_text'].map(lambda x: sha256_text(normalize_for_match(x)))
    return data

def load_benchmark() -> pd.DataFrame:
    if USE_SYNTHETIC_DATA:
        print('USING SYNTHETIC ENGINEERING FIXTURE — NOT SCIENTIFIC DATA')
        return canonicalize_benchmark(synthetic_benchmark())
    if not INPUT_WORKBOOK.exists():
        raise FileNotFoundError(
            f'Benchmark not found: {INPUT_WORKBOOK}. Place the 42-row workbook there or set UNLEARNING_INPUT_WORKBOOK.'
        )
    return canonicalize_benchmark(pd.read_excel(INPUT_WORKBOOK, sheet_name=INPUT_SHEET))

BENCHMARK_LOAD_ERROR = None
try:
    BENCHMARK = load_benchmark()
except Exception as exc:
    BENCHMARK = None
    BENCHMARK_LOAD_ERROR = f'{type(exc).__name__}: {exc}'
    print('BENCHMARK LOAD FAILED:', BENCHMARK_LOAD_ERROR)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [43]:
def token_set(value: Any) -> set[str]:
    return set(re.findall(r'[a-z0-9]+', normalize_for_match(value)))

def token_jaccard(left: Any, right: Any) -> float:
    a, b = token_set(left), token_set(right)
    return len(a & b) / len(a | b) if (a | b) else 1.0

def near_duplicate_pairs(data: pd.DataFrame, threshold: float = NEAR_DUPLICATE_JACCARD_THRESHOLD) -> pd.DataFrame:
    records = data[['row_id','target_text','document_title']].to_dict('records')
    rows = []
    for i in range(len(records)):
        for j in range(i+1, len(records)):
            score = token_jaccard(records[i]['target_text'], records[j]['target_text'])
            if score >= threshold:
                rows.append({
                    'row_id_a': records[i]['row_id'], 'row_id_b': records[j]['row_id'],
                    'document_a': records[i]['document_title'], 'document_b': records[j]['document_title'],
                    'token_jaccard': score, 'text_a': records[i]['target_text'], 'text_b': records[j]['target_text'],
                })
    return pd.DataFrame(rows).sort_values('token_jaccard', ascending=False) if rows else pd.DataFrame()

def audit_benchmark(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    exact = data[data.duplicated('normalized_target_sha256', keep=False)]
    near = near_duplicate_pairs(data)
    checks = [
        {'check':'unique row_id','passed':data['row_id'].is_unique,'observed':data['row_id'].nunique(),'expected':len(data),'severity':'error'},
        {'check':'nonempty target','passed':data['target_text'].str.len().gt(0).all(),'observed':data['target_text'].str.len().gt(0).sum(),'expected':len(data),'severity':'error'},
        {'check':'five context markers','passed':data['marker_count'].ge(5).all(),'observed':data['marker_count'].ge(5).sum(),'expected':len(data),'severity':'error'},
        {'check':'target-context match','passed':data['context_target_similarity'].ge(TARGET_CONTEXT_MATCH_THRESHOLD).all(),'observed':data['context_target_similarity'].min(),'expected':f'>={TARGET_CONTEXT_MATCH_THRESHOLD}','severity':'error'},
        {'check':'no exact target duplicates','passed':exact.empty,'observed':len(exact),'expected':0,'severity':'error'},
        {'check':'expected row count','passed':USE_SYNTHETIC_DATA or len(data)==EXPECTED_BENCHMARK_ROWS,'observed':len(data),'expected':EXPECTED_BENCHMARK_ROWS,'severity':'warning'},
        {'check':'historical labels complete','passed':data['gold_historical'].notna().all(),'observed':data['gold_historical'].notna().sum(),'expected':len(data),'severity':'error'},
    ]
    return pd.DataFrame(checks), near

if BENCHMARK is not None:
    BENCHMARK_AUDIT, NEAR_DUPLICATES = audit_benchmark(BENCHMARK)
    display(BENCHMARK_AUDIT)
    display(BENCHMARK.groupby(['document_title','gold_historical'], dropna=False).size().rename('rows').reset_index())
    if not NEAR_DUPLICATES.empty:
        display(NEAR_DUPLICATES)
else:
    BENCHMARK_AUDIT, NEAR_DUPLICATES = pd.DataFrame(), pd.DataFrame()

def assert_benchmark_ready(data: Optional[pd.DataFrame]) -> pd.DataFrame:
    if data is None:
        raise RuntimeError(f'Benchmark unavailable: {BENCHMARK_LOAD_ERROR}')
    failed = BENCHMARK_AUDIT[(~BENCHMARK_AUDIT['passed']) & BENCHMARK_AUDIT['severity'].eq('error')]
    if not failed.empty:
        raise RuntimeError('Benchmark integrity checks failed:\n' + failed.to_string(index=False))
    return data

,check,passed,observed,expected,severity
0,unique row_id,True,42.0,42,error
1,nonempty target,True,42.0,42,error
2,five context markers,False,40.0,42,error
3,target-context match,True,1.0,>=0.97,error
4,no exact target duplicates,True,0.0,0,error
5,expected row count,True,42.0,42,warning
6,historical labels complete,True,42.0,42,error


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,document_title,gold_historical,rows
0,"Hurricane Katrina: GAO’s Preliminary Observations Regarding Preparedness, Response, and Recovery",No,4
1,"Hurricane Katrina: GAO’s Preliminary Observations Regarding Preparedness, Response, and Recovery",Yes,10
2,Lessons Learned: EPA’s Response to Hurricane Katrina,No,5
3,Lessons Learned: EPA’s Response to Hurricane Katrina,Yes,16
4,The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold,No,1
5,The Katrina Effect on American Preparedness — A report on the lessons Americans learned in watching the Katrina catastrophe unfold,Yes,6


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [44]:
DEFINITION_GOLD_COLUMN = {
    'D0_none':'gold_historical',
    'D1_old_broad':'gold_old_definition',
    'D2_current_strict':'gold_current_definition',
    'D3_provisional_adaptive':'gold_final_definition',
}
GOLD_AVAILABILITY = pd.DataFrame([
    {
        'definition_id': definition_id,
        'preferred_gold_column': column,
        'complete': bool(BENCHMARK is not None and column in BENCHMARK and BENCHMARK[column].notna().all()),
        'non_null_rows': int(BENCHMARK[column].notna().sum()) if BENCHMARK is not None and column in BENCHMARK else 0,
    }
    for definition_id, column in DEFINITION_GOLD_COLUMN.items()
])
display(GOLD_AVAILABILITY)

DATASET_HASH_COLUMNS = [
    'row_id','target_text','document_title','pdf_page','section_heading','paragraph_order',
    'context_previous_2','context_previous_1','context_target','context_next_1','context_next_2',
    'gold_historical','gold_old_definition','gold_current_definition','gold_final_definition',
    'gold_target','gold_agency'
]
DATASET_SHA256 = None
if BENCHMARK is not None:
    canonical = BENCHMARK[DATASET_HASH_COLUMNS].fillna('').astype(str).to_csv(index=False, lineterminator='\n')
    DATASET_SHA256 = sha256_text(canonical)
    BENCHMARK.to_csv(ARTIFACTS_ROOT / 'benchmark_canonical.csv', index=False)
    with contextlib.suppress(Exception):
        BENCHMARK.to_parquet(ARTIFACTS_ROOT / 'benchmark_canonical.parquet', index=False)
    manifest = {
        'dataset_sha256': DATASET_SHA256,
        'source_path': str(INPUT_WORKBOOK),
        'source_file_sha256': sha256_file(INPUT_WORKBOOK),
        'sheet': INPUT_SHEET,
        'rows': len(BENCHMARK),
        'columns_hashed': DATASET_HASH_COLUMNS,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    (ARTIFACTS_ROOT / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('DATASET SHA-256:', DATASET_SHA256)

,definition_id,preferred_gold_column,complete,non_null_rows
0,D0_none,gold_historical,True,42
1,D1_old_broad,gold_old_definition,False,0
2,D2_current_strict,gold_current_definition,False,0
3,D3_provisional_adaptive,gold_final_definition,False,0


DATASET SHA-256: ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 8. Experimental condition registry

In [45]:
PromptStyle = Literal['P1_direct','P2_simple_definition','P3_evidence_checklist']
ContextLevel = Literal['C1_target','C2_metadata','C3_plusminus1','C4_plusminus2']
Workflow = Literal['W1_one_stage','W2_binary_first']
Stage = Literal['joint','binary','target']

@dataclass(frozen=True)
class ConditionSpec:
    condition_id: str
    phase: str
    definition_id: str
    prompt_style: PromptStyle
    context_id: ContextLevel
    workflow: Workflow
    description: str

PHASE1_CONDITIONS = [
    ConditionSpec('P1_DIRECT','phase1','D0_none','P1_direct','C1_target','W1_one_stage','Direct task; no supplied definition.'),
    ConditionSpec('P2_D1_OLD','phase1','D1_old_broad','P2_simple_definition','C1_target','W1_one_stage','Old definition + simple structured classification.'),
    ConditionSpec('P2_D2_CURRENT','phase1','D2_current_strict','P2_simple_definition','C1_target','W1_one_stage','Current definition + simple structured classification.'),
    ConditionSpec('P2_D3_ADAPTIVE','phase1','D3_provisional_adaptive','P2_simple_definition','C1_target','W1_one_stage','Adaptive candidate + simple structured classification.'),
    ConditionSpec('P3_D1_OLD','phase1','D1_old_broad','P3_evidence_checklist','C1_target','W1_one_stage','Old definition + explicit evidence checklist.'),
    ConditionSpec('P3_D2_CURRENT','phase1','D2_current_strict','P3_evidence_checklist','C1_target','W1_one_stage','Current definition + explicit evidence checklist.'),
    ConditionSpec('P3_D3_ADAPTIVE','phase1','D3_provisional_adaptive','P3_evidence_checklist','C1_target','W1_one_stage','Adaptive candidate + explicit evidence checklist.'),
]
PHASE1_REGISTRY = pd.DataFrame([asdict(x) for x in PHASE1_CONDITIONS])
display(PHASE1_REGISTRY)

,condition_id,phase,definition_id,prompt_style,context_id,workflow,description
0,P1_DIRECT,phase1,D0_none,P1_direct,C1_target,W1_one_stage,Direct task; no supplied definition.
1,P2_D1_OLD,phase1,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,Old definition + simple structured classification.
2,P2_D2_CURRENT,phase1,D2_current_strict,P2_simple_definition,C1_target,W1_one_stage,Current definition + simple structured classification.
3,P2_D3_ADAPTIVE,phase1,D3_provisional_adaptive,P2_simple_definition,C1_target,W1_one_stage,Adaptive candidate + simple structured classification.
4,P3_D1_OLD,phase1,D1_old_broad,P3_evidence_checklist,C1_target,W1_one_stage,Old definition + explicit evidence checklist.
5,P3_D2_CURRENT,phase1,D2_current_strict,P3_evidence_checklist,C1_target,W1_one_stage,Current definition + explicit evidence checklist.
6,P3_D3_ADAPTIVE,phase1,D3_provisional_adaptive,P3_evidence_checklist,C1_target,W1_one_stage,Adaptive candidate + explicit evidence checklist.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Prompt construction and leakage audit

In [46]:
TARGET_CODEBOOK_TEXT = """
TARGET CATEGORY — assign exactly one only when unlearning_present is true.
- leadership: prior leadership, command, authority, accountability, or decision-right arrangement is displaced.
- laws_plans_policies: prior law, policy, doctrine, plan, formal rule, standard, or operating policy is displaced.
- capabilities: prior technical system, procedure, information process, organizational capability, operational method, or professional routine is displaced.
- funds_resources: prior funding rule, budget logic, resource-distribution practice, procurement arrangement, or material-resource policy is displaced.
- misc_organizational: qualifying organizational arrangement not represented above; use sparingly.
- none: required when unlearning_present is false.
""".strip()

EVIDENCE_RULES_TEXT = """
STRUCTURED EVIDENCE RULES
- Return only registered fields. Do not provide narrative explanation or hidden chain-of-thought.
- Every quote must be an exact short excerpt from one supplied source block or null.
- Declare the correct source_scope.
- Do not invent a prior state, failure, change, agency, or target.
- Context may resolve antecedents or background, but a neighbor cannot independently make TARGET positive.
- A positive decision must include TARGET evidence of the decisive departure or reconfiguration.
- For No, set target_type='none' and agency=null in joint output.
- Set needs_human_review=true for ambiguous boundaries, mixed mechanisms, or uncertain target/agency.
""".strip()

DIRECT_TASK_TEXT = """
Classify whether the TARGET contains organizational unlearning in a government organization using ordinary understanding. Extract exact textual evidence before deciding.
""".strip()
SIMPLE_TASK_TEXT = """
Classify whether TARGET satisfies the supplied operational definition. Return the decision, change mechanism, structured evidence, target, agency, confidence, and review flag.
""".strip()
CHECKLIST_TASK_TEXT = """
Apply the definition with an explicit evidence checklist. Identify separately: (1) prior state; (2) inadequacy/failure; (3) departure or durable reconfiguration in TARGET. Then decide and assign mechanism, target, agency, confidence, and review flag.
""".strip()
STAGE2_TASK_TEXT = """
Stage 1 already made a positive binary judgment. Do not reconsider it. Assign target category and agency, grounded in one exact excerpt.
""".strip()

def schema_id_for(prompt_style: PromptStyle, stage: Stage) -> str:
    checklist = prompt_style == 'P3_evidence_checklist'
    if stage == 'joint':
        return 'checklist_joint' if checklist else 'simple_joint'
    if stage == 'binary':
        return 'checklist_binary' if checklist else 'simple_binary'
    return 'stage2_target'

def source_blocks_for_context(row: pd.Series, context_id: ContextLevel) -> dict[str,str]:
    blocks = {'target': normalize_space(row['target_text'])}
    if context_id in {'C2_metadata','C3_plusminus1','C4_plusminus2'}:
        blocks['metadata'] = (
            f"Document title: {normalize_space(row['document_title'])}\n"
            f"Section heading: {normalize_space(row['section_heading'])}\n"
            f"PDF page: {normalize_space(row.get('pdf_page'))}"
        )
    if context_id in {'C3_plusminus1','C4_plusminus2'}:
        blocks['previous_1'] = normalize_space(row['context_previous_1'])
        blocks['next_1'] = normalize_space(row['context_next_1'])
    if context_id == 'C4_plusminus2':
        blocks['previous_2'] = normalize_space(row['context_previous_2'])
        blocks['next_2'] = normalize_space(row['context_next_2'])
    return blocks

def render_variable_input(row: pd.Series, context_id: ContextLevel) -> str:
    blocks = source_blocks_for_context(row, context_id)
    sections = [f"ROW ID: {row['row_id']}"]
    if blocks.get('metadata'):
        sections.append('<METADATA>\n'+blocks['metadata']+'\n</METADATA>')
    if blocks.get('previous_2'):
        sections.append('<PREVIOUS_2>\n'+blocks['previous_2']+'\n</PREVIOUS_2>')
    if blocks.get('previous_1'):
        sections.append('<PREVIOUS_1>\n'+blocks['previous_1']+'\n</PREVIOUS_1>')
    sections.append('<TARGET>\n'+blocks['target']+'\n</TARGET>')
    if blocks.get('next_1'):
        sections.append('<NEXT_1>\n'+blocks['next_1']+'\n</NEXT_1>')
    if blocks.get('next_2'):
        sections.append('<NEXT_2>\n'+blocks['next_2']+'\n</NEXT_2>')
    return '\n\n'.join(sections)

@dataclass(frozen=True)
class PromptPackage:
    system_prompt: str
    user_prompt: str
    schema_id: str
    schema_model: Type[BaseModel]
    schema_sha256: str
    prompt_sha256: str
    cache_key: str
    source_blocks: dict[str,str]

def build_system_prompt(condition: ConditionSpec, stage: Stage) -> str:
    sections = [
        'You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.',
        'Classify one TARGET paragraph per request. Never infer or reproduce a human label. Return only registered structured output.',
    ]
    if condition.prompt_style == 'P1_direct':
        sections.append(DIRECT_TASK_TEXT)
    else:
        sections.append('OPERATIONAL DEFINITION\n\n'+DEFINITIONS[condition.definition_id].operational_text)
        sections.append(CHECKLIST_TASK_TEXT if condition.prompt_style == 'P3_evidence_checklist' else SIMPLE_TASK_TEXT)
    if stage in {'joint','target'}:
        sections.append(TARGET_CODEBOOK_TEXT)
    sections.append(EVIDENCE_RULES_TEXT)
    if stage == 'binary':
        sections.append('This is Stage 1. Return binary/evidence fields only; do not assign target or agency.')
    elif stage == 'target':
        sections.append(STAGE2_TASK_TEXT)
    sections.append(f'SCHEMA ID: {schema_id_for(condition.prompt_style, stage)}')
    sections.append(f'PROTOCOL VERSION: {PROTOCOL_VERSION}')
    return '\n\n'.join(sections)

def build_prompt_package(row: pd.Series, condition: ConditionSpec, stage: Stage='joint', stage1_output: Optional[dict[str,Any]]=None) -> PromptPackage:
    schema_id = schema_id_for(condition.prompt_style, stage)
    schema_model = SCHEMA_MODELS[schema_id]
    system_prompt = build_system_prompt(condition, stage)
    user_prompt = render_variable_input(row, condition.context_id)
    if stage == 'target':
        if stage1_output is None:
            raise ValueError('Stage 2 requires Stage-1 output.')
        user_prompt += '\n\n<STAGE_1_STRUCTURED_RESULT>\n'+json.dumps(stage1_output, ensure_ascii=False, sort_keys=True)+'\n</STAGE_1_STRUCTURED_RESULT>'
    schema_hash = sha256_text(canonical_json(schema_model.model_json_schema()))
    prompt_hash = sha256_text(canonical_json({'system':system_prompt,'user':user_prompt,'schema':schema_hash}))
    return PromptPackage(
        system_prompt, user_prompt, schema_id, schema_model, schema_hash, prompt_hash,
        f'{PROTOCOL_VERSION}:{condition.definition_id}:{condition.prompt_style}:{stage}:{schema_id}',
        source_blocks_for_context(row, condition.context_id),
    )

In [47]:
FORBIDDEN_PROMPT_TERMS = [
    'gold_historical','gold_old_definition','gold_current_definition','gold_final_definition',
    'gold_target','gold_agency','original rationale','labeled examples'
]

def audit_prompt_package(package: PromptPackage, row: pd.Series, condition: ConditionSpec) -> dict[str,Any]:
    combined = (package.system_prompt+'\n'+package.user_prompt).casefold()
    target = normalize_for_match(row['target_text'])
    user = normalize_for_match(package.user_prompt)
    return {
        'condition_id':condition.condition_id,'definition_id':condition.definition_id,
        'prompt_style':condition.prompt_style,'context_id':condition.context_id,
        'schema_id':package.schema_id,'target_occurrences_normalized':user.count(target),
        'contains_forbidden_term':any(term.casefold() in combined for term in FORBIDDEN_PROMPT_TERMS),
        'contains_definition':bool(DEFINITIONS[condition.definition_id].operational_text and DEFINITIONS[condition.definition_id].operational_text in package.system_prompt),
        'contains_checklist':'explicit evidence checklist' in package.system_prompt,
        'prompt_sha256':package.prompt_sha256,'system_chars':len(package.system_prompt),'user_chars':len(package.user_prompt),
    }

PROMPT_AUDIT = pd.DataFrame()
if BENCHMARK is not None:
    PROMPT_AUDIT = pd.DataFrame([
        audit_prompt_package(build_prompt_package(BENCHMARK.iloc[0], c), BENCHMARK.iloc[0], c)
        for c in PHASE1_CONDITIONS
    ])
    assert PROMPT_AUDIT['target_occurrences_normalized'].eq(1).all()
    assert not PROMPT_AUDIT['contains_forbidden_term'].any()
    display(PROMPT_AUDIT)

,condition_id,definition_id,prompt_style,context_id,schema_id,target_occurrences_normalized,contains_forbidden_term,contains_definition,contains_checklist,prompt_sha256,system_chars,user_chars
0,P1_DIRECT,D0_none,P1_direct,C1_target,simple_joint,1,False,False,False,bce62c373728c42072b4c1232e0fa037f986e4714070dc7904d3fab48603c9e9,1936,405
1,P2_D1_OLD,D1_old_broad,P2_simple_definition,C1_target,simple_joint,1,False,True,False,521d9a6d06fd87d91d09d2cfbd9a4d1b3879eaaab592ec3f98993f6e2e8dfa36,2676,405
2,P2_D2_CURRENT,D2_current_strict,P2_simple_definition,C1_target,simple_joint,1,False,True,False,2cb738b4484baadea3049234425004d5d8b7c44ad073b7d3176c83f4c5365493,2707,405
3,P2_D3_ADAPTIVE,D3_provisional_adaptive,P2_simple_definition,C1_target,simple_joint,1,False,True,False,3906c1026d93206f9c3bdd2db958b3caffbbbb703f3bec4020b277a167a8237b,3196,405
4,P3_D1_OLD,D1_old_broad,P3_evidence_checklist,C1_target,checklist_joint,1,False,True,True,b195f908d44840a6555d8d0c2139d1a3592dd8743ff83094859a2e2edfef88ef,2754,405
5,P3_D2_CURRENT,D2_current_strict,P3_evidence_checklist,C1_target,checklist_joint,1,False,True,True,5da0db4ea68a79af9ed8ac0db52a0c4194c3e2da5e06f63e7e94403b1550f480,2785,405
6,P3_D3_ADAPTIVE,D3_provisional_adaptive,P3_evidence_checklist,C1_target,checklist_joint,1,False,True,True,8d455c73bec4e34107e8ee5f28c84408680662ab12a460881cfa4256c10643fe,3274,405


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 10. Balanced scheduling, append-only logs, and deterministic run keys

In [48]:
@dataclass(frozen=True)
class TaskSpec:
    phase: str
    condition_id: str
    row_id: str
    seed: int
    repeat_id: str
    stage: Stage
    sequence: int

def conditions_by_id(conditions: Sequence[ConditionSpec]) -> dict[str,ConditionSpec]:
    mapping = {condition.condition_id: condition for condition in conditions}
    if len(mapping) != len(conditions):
        raise ValueError('Condition IDs must be unique within a phase.')
    return mapping

def blocked_condition_order(row_ids: Sequence[str], condition_ids: Sequence[str], seed: int) -> list[tuple[str,str,int]]:
    rng = random.Random(stable_int_seed(PROTOCOL_VERSION, seed, 'schedule'))
    shuffled_rows = list(row_ids)
    rng.shuffle(shuffled_rows)
    rows = []
    sequence = 0
    for row_index, row_id in enumerate(shuffled_rows):
        local_conditions = list(condition_ids)
        local_rng = random.Random(stable_int_seed(PROTOCOL_VERSION, seed, row_id, 'condition_order'))
        local_rng.shuffle(local_conditions)
        # Rotate to reduce provider-time confounding beyond random shuffling.
        offset = row_index % max(1, len(local_conditions))
        local_conditions = local_conditions[offset:] + local_conditions[:offset]
        for condition_id in local_conditions:
            rows.append((str(row_id), condition_id, sequence))
            sequence += 1
    return rows

def make_task_schedule(data: pd.DataFrame, conditions: Sequence[ConditionSpec], seeds: Sequence[int], phase: str, stage: Stage='joint') -> list[TaskSpec]:
    tasks = []
    for seed in seeds:
        ordered = blocked_condition_order(data['row_id'].astype(str).tolist(), [c.condition_id for c in conditions], seed)
        for row_id, condition_id, sequence in ordered:
            tasks.append(TaskSpec(
                phase=phase, condition_id=condition_id, row_id=row_id, seed=seed,
                repeat_id=f'seed_{seed}_{RUN_REPLICATE_ID}', stage=stage, sequence=sequence,
            ))
    return tasks

def provider_phase_directory(phase: str, provider: str) -> Path:
    path = RUNS_ROOT / phase / provider
    path.mkdir(parents=True, exist_ok=True)
    return path

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix+'.tmp')
    temp.write_text(text, encoding='utf-8')
    temp.replace(path)

def atomic_write_dataframe_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix+'.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)

def append_jsonl(path: Path, record: dict[str,Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False, default=str)+'\n')

def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    records = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Malformed JSONL at {path}:{line_number}: {exc}') from exc
    return pd.DataFrame(records)

def latest_records_by_run_key(path: Path) -> dict[str,dict[str,Any]]:
    latest = {}
    if not path.exists():
        return latest
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                if record.get('run_key'):
                    latest[record['run_key']] = record
    return latest

def successful_run_keys(path: Path) -> set[str]:
    return {key for key, record in latest_records_by_run_key(path).items() if record.get('status') == 'ok'}

def make_run_key(provider_config: ProviderModelConfig, task: TaskSpec, condition: ConditionSpec, package: PromptPackage) -> str:
    payload = {
        'protocol_version':PROTOCOL_VERSION,'dataset_sha256':DATASET_SHA256,
        'provider':provider_config.provider,'model':provider_config.model_id,
        'model_config_sha256':model_config_hash(provider_config),'phase':task.phase,
        'condition_id':task.condition_id,'definition_id':condition.definition_id,
        'definition_sha256':sha256_text(DEFINITIONS[condition.definition_id].operational_text),
        'prompt_style':condition.prompt_style,'context_id':condition.context_id,
        'workflow':condition.workflow,'stage':task.stage,'row_id':task.row_id,
        'seed':task.seed,'repeat_id':task.repeat_id,'schema_sha256':package.schema_sha256,
        'prompt_sha256':package.prompt_sha256,
    }
    return sha256_text(canonical_json(payload))

## 11. Provider result contract, retries, and deterministic mock

In [49]:
@dataclass
class ProviderCallResult:
    provider: str
    requested_model: str
    returned_model: Optional[str]
    request_id: Optional[str]
    raw_response_text: str
    parsed_output: dict[str,Any]
    input_tokens: Optional[int]
    output_tokens: Optional[int]
    total_tokens: Optional[int]
    reasoning_tokens: Optional[int]
    cached_input_tokens: Optional[int]
    cache_creation_input_tokens: Optional[int]
    stop_reason: Optional[str]
    latency_seconds: float
    accepted_parameters: dict[str,Any]
    provider_metadata: dict[str,Any] = field(default_factory=dict)

class ProviderProtocolError(RuntimeError): pass
class ProviderConfigurationError(ProviderProtocolError): pass
class ProviderTransientError(ProviderProtocolError): pass

def safe_attr(obj: Any, path: str, default: Any=None) -> Any:
    current = obj
    for part in path.split('.'):
        if current is None:
            return default
        if isinstance(current, dict):
            current = current.get(part, default)
        elif part.isdigit() and isinstance(current, (list,tuple)):
            index = int(part)
            current = current[index] if index < len(current) else default
        else:
            current = getattr(current, part, default)
    return current

def exception_status_code(exc: Exception) -> Optional[int]:
    for path in ['status_code','response.status_code','code']:
        value = safe_attr(exc, path)
        with contextlib.suppress(TypeError,ValueError):
            if value is not None:
                return int(value)
    return None

def error_classification(exc: Exception) -> str:
    status = exception_status_code(exc)
    text = (type(exc).__name__+' '+str(exc)).casefold()
    if status in {408,409,425,429} or (status is not None and status >= 500):
        return 'transient'
    if any(x in text for x in ['timeout','connection','temporar','rate limit','overloaded']):
        return 'transient'
    if status in {400,401,403,404,422}:
        return 'configuration'
    return 'unknown'

def call_with_retry(function: Callable[[],ProviderCallResult], seed: int) -> ProviderCallResult:
    rng = random.Random(stable_int_seed(PROTOCOL_VERSION, seed, 'retry'))
    for attempt in range(1, MAX_RETRY_ATTEMPTS+1):
        try:
            return function()
        except Exception as exc:
            if error_classification(exc) != 'transient' or attempt == MAX_RETRY_ATTEMPTS:
                raise
            delay = BASE_RETRY_SECONDS * 2**(attempt-1) + rng.uniform(0, BASE_RETRY_SECONDS)
            print(f'Transient error {attempt}/{MAX_RETRY_ATTEMPTS}; retry in {delay:.2f}s: {exc}')
            time.sleep(delay)
    raise RuntimeError('Retry loop ended unexpectedly.')

In [50]:
def extract_tagged_target(user_prompt: str) -> str:
    match = re.search(r'<TARGET>\s*(.*?)\s*</TARGET>', user_prompt, flags=re.S)
    return normalize_space(match.group(1)) if match else ''

def first_matching_quote(text: str, patterns: Sequence[str]) -> Optional[str]:
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return normalize_space(match.group(0))
    return None

def mock_decision(package: PromptPackage, schema_model: Type[BaseModel], seed: int) -> dict[str,Any]:
    target = extract_tagged_target(package.user_prompt)
    lower = target.casefold()
    adaptive = 'ROUTE B — ADAPTIVE RECONFIGURATION' in package.system_prompt
    old = 'BROAD HISTORICAL OPERATIONALIZATION' in package.system_prompt
    subtractive = any(x in lower for x in ['replace','eliminat','abandon','remove','move away','redistribut','no longer','fundamentally'])
    reconfiguration = any(x in lower for x in ['protocol','defined agency responsibilities','formal liaison','standard operating procedure','integrated operational plan'])
    capacity_only = any(x in lower for x in ['additional staff','mobile laborator','refresher training','user guide','equipment']) and not reconfiguration
    diagnosis_only = any(x in lower for x in ['documented','could have been better','experienced problems']) and not (subtractive or reconfiguration)
    positive = subtractive
    mode = 'subtractive_discontinuity' if subtractive else 'none'
    change_type = 'replace' if 'replace' in lower else ('redistribute_authority' if 'redistribut' in lower else ('remove' if 'eliminat' in lower else 'no_change'))
    if adaptive and reconfiguration and not capacity_only:
        positive, mode, change_type = True, 'adaptive_reconfiguration', 'revise_protocol_or_standard'
    elif old and (reconfiguration or capacity_only):
        positive, mode = True, 'technical_realignment'
        change_type = 'revise_protocol_or_standard' if reconfiguration else 'add_capacity_only'
    elif capacity_only:
        positive, change_type = False, 'add_capacity_only'
    elif diagnosis_only:
        positive, change_type = False, 'diagnosis_only'
    departure = first_matching_quote(target,[r'replaced[^.]*',r'redistributed[^.]*',r'eliminated[^.]*',r'established[^.]*protocol[^.]*',r'added[^.]*',r'documented[^.]*'])
    prior = first_matching_quote(package.user_prompt,[r'layered approach delayed aid',r'central approval caused delay',r'duplicated work followed unclear roles',r'analytical capacity was insufficient',r'Some staff misunderstood the procedure'])
    target_type: TargetType = 'none'
    if positive:
        if any(x in lower for x in ['authority','commander','leadership']): target_type='leadership'
        elif any(x in lower for x in ['doctrine','policy','framework']): target_type='laws_plans_policies'
        elif any(x in lower for x in ['protocol','procedure','laborator']): target_type='capabilities'
        else: target_type='misc_organizational'
    confidence = 0.91 if subtractive else (0.78 if positive else 0.82)
    review = bool(reconfiguration or capacity_only)
    if schema_model in {SimpleJointOutput,SimpleBinaryOutput}:
        evidence=[]
        if prior: evidence.append({'element':'prior_state','quote':prior,'source_scope':'previous_1'})
        if departure: evidence.append({'element':'departure_or_reconfiguration','quote':departure,'source_scope':'target'})
        payload={'unlearning_present':positive,'unlearning_mode':mode,'change_type':change_type,'evidence':evidence,'confidence':confidence,'needs_human_review':review}
        if schema_model is SimpleJointOutput: payload.update({'target_type':target_type,'agency':'Synthetic agency' if positive else None})
    elif schema_model in {ChecklistJointOutput,ChecklistBinaryOutput}:
        payload={
            'prior_state':{'identified':bool(prior),'quote':prior,'source_scope':'previous_1' if prior else 'absent'},
            'inadequacy_or_failure':{'identified':bool(prior),'quote':prior,'source_scope':'previous_1' if prior else 'absent'},
            'departure_or_reconfiguration':{'identified':bool(departure),'quote':departure,'source_scope':'target' if departure else 'absent'},
            'unlearning_mode':mode,'change_type':change_type,
            'all_required_elements_present':bool(prior and departure),
            'missing_elements':[name for name,present in [('prior_state',bool(prior)),('inadequacy_or_failure',bool(prior)),('departure_or_reconfiguration',bool(departure))] if not present],
            'unlearning_present':positive,'confidence':confidence,'needs_human_review':review,
        }
        if schema_model is ChecklistJointOutput: payload.update({'target_type':target_type,'agency':'Synthetic agency' if positive else None})
    elif schema_model is Stage2TargetOutput:
        payload={'target_type':target_type if target_type!='none' else 'misc_organizational','agency':'Synthetic agency','target_evidence_quote':departure,'target_evidence_scope':'target' if departure else 'absent','confidence':confidence,'needs_human_review':review}
    else:
        raise ValueError(f'Unsupported mock schema: {schema_model}')
    return schema_model.model_validate(payload).model_dump()

def call_mock_provider(config: ProviderModelConfig, package: PromptPackage, seed: int) -> ProviderCallResult:
    started=time.perf_counter(); parsed=mock_decision(package,package.schema_model,seed); raw=json.dumps(parsed,ensure_ascii=False)
    input_tokens=len((package.system_prompt+' '+package.user_prompt).split()); output_tokens=len(raw.split())
    return ProviderCallResult(config.provider,config.model_id,'mock-deterministic-v1',f'mock_{uuid.uuid4().hex}',raw,parsed,input_tokens,output_tokens,input_tokens+output_tokens,0,0,0,'mock_complete',time.perf_counter()-started,{'temperature':config.temperature,'mock':True})

## 12. Provider-native structured-output adapters

The adapters below deliberately do not share execution state. Each provider has its own client, smoke-test cell, result directory, JSONL log, CSV snapshot, error log, and status file. A provider configuration error is recorded and stops that provider only.

The protocol does not silently fall back from structured output to free-form JSON, remove temperature 0, alter the requested reasoning setting, or substitute a different model. Any such change must become a new registered model configuration.

In [51]:
_OPENAI_CLIENT = None

def get_openai_client():
    global _OPENAI_CLIENT
    if _OPENAI_CLIENT is None:
        from openai import OpenAI
        _OPENAI_CLIENT = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    return _OPENAI_CLIENT

def call_openai_provider(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    assert_api_execution_allowed('openai')
    client = get_openai_client()
    request_parameters: dict[str, Any] = {
        'model': config.model_id,
        'input': [
            {
                'role': 'developer',
                'content': [
                    {'type': 'input_text', 'text': package.system_prompt},
                ],
            },
            {
                'role': 'user',
                'content': [
                    {'type': 'input_text', 'text': package.user_prompt},
                ],
            },
        ],
        'text_format': package.schema_model,
        'reasoning': config.reasoning_payload,
        'temperature': config.temperature,
        'max_output_tokens': (
            MAX_STAGE2_OUTPUT_TOKENS
            if package.schema_id == 'stage2_target'
            else config.max_output_tokens
        ),
        'prompt_cache_key': package.cache_key,
        'store': False,
    }
    if PASS_PROVIDER_NATIVE_SEED:
        # The Responses API may not expose a seed for every model. This field is
        # intentionally opt-in and must pass the provider smoke test.
        request_parameters['seed'] = seed

    started = time.perf_counter()
    response = client.responses.parse(**request_parameters)
    latency = time.perf_counter() - started
    parsed = response.output_parsed
    if parsed is None:
        raise ProviderProtocolError('OpenAI returned no output_parsed object.')
    parsed_dict = parsed.model_dump() if isinstance(parsed, BaseModel) else dict(parsed)
    validated = package.schema_model.model_validate(parsed_dict).model_dump()

    input_tokens = safe_attr(response, 'usage.input_tokens')
    output_tokens = safe_attr(response, 'usage.output_tokens')
    total_tokens = safe_attr(response, 'usage.total_tokens')
    reasoning_tokens = safe_attr(response, 'usage.output_tokens_details.reasoning_tokens', 0)
    cached_tokens = safe_attr(response, 'usage.input_tokens_details.cached_tokens', 0)
    cache_write_tokens = safe_attr(response, 'usage.input_tokens_details.cache_write_tokens', 0)
    raw_text = safe_attr(response, 'output_text') or canonical_json(validated)

    return ProviderCallResult(
        provider='openai',
        requested_model=config.model_id,
        returned_model=safe_attr(response, 'model'),
        request_id=safe_attr(response, 'id'),
        raw_response_text=raw_text,
        parsed_output=validated,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        reasoning_tokens=reasoning_tokens,
        cached_input_tokens=cached_tokens,
        cache_creation_input_tokens=cache_write_tokens,
        stop_reason=safe_attr(response, 'status'),
        latency_seconds=latency,
        accepted_parameters={
            'temperature': config.temperature,
            'reasoning': config.reasoning_payload,
            'max_output_tokens': request_parameters['max_output_tokens'],
            'prompt_cache_key': package.cache_key,
            'store': False,
            'provider_seed_sent': PASS_PROVIDER_NATIVE_SEED,
        },
        provider_metadata={
            'sdk_version': package_version('openai'),
            'incomplete_details': safe_attr(response, 'incomplete_details'),
            'usage': safe_attr(response, 'usage'),
        },
    )

In [52]:
_ANTHROPIC_CLIENT = None

def get_anthropic_client():
    global _ANTHROPIC_CLIENT
    if _ANTHROPIC_CLIENT is None:
        from anthropic import Anthropic
        _ANTHROPIC_CLIENT = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
    return _ANTHROPIC_CLIENT

def call_anthropic_provider(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    assert_api_execution_allowed('anthropic')
    client = get_anthropic_client()
    request_parameters: dict[str, Any] = {
        'model': config.model_id,
        'max_tokens': (
            MAX_STAGE2_OUTPUT_TOKENS
            if package.schema_id == 'stage2_target'
            else config.max_output_tokens
        ),
        'temperature': config.temperature,
        'cache_control': {'type': 'ephemeral'},
        'system': package.system_prompt,
        'messages': [{'role': 'user', 'content': package.user_prompt}],
        'output_format': package.schema_model,
    }
    # Haiku 4.5's lowest comparable setting is extended thinking omitted.
    if config.reasoning_payload is not None:
        request_parameters['thinking'] = config.reasoning_payload

    started = time.perf_counter()
    response = client.messages.parse(**request_parameters)
    latency = time.perf_counter() - started
    parsed = safe_attr(response, 'parsed_output')
    if parsed is None:
        raise ProviderProtocolError('Anthropic returned no parsed_output object.')
    parsed_dict = parsed.model_dump() if isinstance(parsed, BaseModel) else dict(parsed)
    validated = package.schema_model.model_validate(parsed_dict).model_dump()

    content_texts = [
        safe_attr(block, 'text', '')
        for block in (safe_attr(response, 'content', []) or [])
        if safe_attr(block, 'type') == 'text'
    ]
    raw_text = '\n'.join(text for text in content_texts if text) or canonical_json(validated)
    input_tokens = safe_attr(response, 'usage.input_tokens')
    output_tokens = safe_attr(response, 'usage.output_tokens')
    total_tokens = (
        (input_tokens or 0) + (output_tokens or 0)
        if input_tokens is not None or output_tokens is not None
        else None
    )

    return ProviderCallResult(
        provider='anthropic',
        requested_model=config.model_id,
        returned_model=safe_attr(response, 'model'),
        request_id=safe_attr(response, 'id'),
        raw_response_text=raw_text,
        parsed_output=validated,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        reasoning_tokens=0,
        cached_input_tokens=safe_attr(response, 'usage.cache_read_input_tokens', 0),
        cache_creation_input_tokens=safe_attr(response, 'usage.cache_creation_input_tokens', 0),
        stop_reason=safe_attr(response, 'stop_reason'),
        latency_seconds=latency,
        accepted_parameters={
            'temperature': config.temperature,
            'thinking': config.reasoning_payload,
            'max_tokens': request_parameters['max_tokens'],
            'cache_control': {'type': 'ephemeral'},
            'provider_seed_sent': False,
        },
        provider_metadata={
            'sdk_version': package_version('anthropic'),
            'usage': safe_attr(response, 'usage'),
            'stop_sequence': safe_attr(response, 'stop_sequence'),
        },
    )

In [53]:
_GEMINI_CLIENT = None

def get_gemini_client():
    global _GEMINI_CLIENT
    if _GEMINI_CLIENT is None:
        from google import genai
        api_key = os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY')
        _GEMINI_CLIENT = genai.Client(api_key=api_key)
    return _GEMINI_CLIENT

def call_gemini_provider(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    assert_api_execution_allowed('gemini')
    from google.genai import types
    client = get_gemini_client()

    thinking_config = types.ThinkingConfig(
        thinking_level=(config.reasoning_payload or {}).get('thinking_level', 'low')
    )
    generate_config = types.GenerateContentConfig(
        system_instruction=package.system_prompt,
        temperature=config.temperature,
        max_output_tokens=(
            MAX_STAGE2_OUTPUT_TOKENS
            if package.schema_id == 'stage2_target'
            else config.max_output_tokens
        ),
        thinking_config=thinking_config,
        response_mime_type='application/json',
        response_json_schema=package.schema_model.model_json_schema(),
    )
    # The Gemini SDK's deterministic seed field is intentionally opt-in because
    # availability can vary by endpoint/model version.
    if PASS_PROVIDER_NATIVE_SEED and hasattr(generate_config, 'seed'):
        generate_config.seed = seed

    started = time.perf_counter()
    response = client.models.generate_content(
        model=config.model_id,
        contents=package.user_prompt,
        config=generate_config,
    )
    latency = time.perf_counter() - started
    raw_text = safe_attr(response, 'text')
    if not raw_text:
        raise ProviderProtocolError('Gemini returned empty response.text.')
    try:
        parsed_dict = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        raise ProviderProtocolError(f'Gemini response was not JSON: {exc}') from exc
    validated = package.schema_model.model_validate(parsed_dict).model_dump()
    usage = safe_attr(response, 'usage_metadata')
    input_tokens = safe_attr(usage, 'prompt_token_count')
    output_tokens = safe_attr(usage, 'candidates_token_count')
    total_tokens = safe_attr(usage, 'total_token_count')
    cached_tokens = safe_attr(usage, 'cached_content_token_count', 0)
    reasoning_tokens = safe_attr(usage, 'thoughts_token_count', 0)
    finish_reason = safe_attr(response, 'candidates.0.finish_reason')

    return ProviderCallResult(
        provider='gemini',
        requested_model=config.model_id,
        returned_model=safe_attr(response, 'model_version') or config.model_id,
        request_id=safe_attr(response, 'response_id'),
        raw_response_text=raw_text,
        parsed_output=validated,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        reasoning_tokens=reasoning_tokens,
        cached_input_tokens=cached_tokens,
        cache_creation_input_tokens=0,
        stop_reason=str(finish_reason) if finish_reason is not None else None,
        latency_seconds=latency,
        accepted_parameters={
            'temperature': config.temperature,
            'thinking_level': (config.reasoning_payload or {}).get('thinking_level'),
            'max_output_tokens': generate_config.max_output_tokens,
            'response_mime_type': 'application/json',
            'provider_seed_sent': bool(PASS_PROVIDER_NATIVE_SEED and hasattr(generate_config, 'seed')),
        },
        provider_metadata={
            'sdk_version': package_version('google-genai'),
            'usage_metadata': usage,
            'prompt_feedback': safe_attr(response, 'prompt_feedback'),
        },
    )

In [54]:
PROVIDER_CALLS: dict[str, Callable[[ProviderModelConfig,PromptPackage,int],ProviderCallResult]] = {
    'openai': call_openai_provider,
    'anthropic': call_anthropic_provider,
    'gemini': call_gemini_provider,
}

def dispatch_provider_call(
    config: ProviderModelConfig,
    package: PromptPackage,
    seed: int,
) -> ProviderCallResult:
    if USE_MOCK_PROVIDER:
        return call_mock_provider(config, package, seed)
    return PROVIDER_CALLS[config.provider](config, package, seed)

PROVIDER_ADAPTER_AUDIT = pd.DataFrame([
    {
        'provider': provider,
        'model_id': config.model_id,
        'adapter': PROVIDER_CALLS[provider].__name__,
        'temperature': config.temperature,
        'reasoning_label': config.reasoning_label,
        'prompt_caching': config.prompt_caching,
        'sdk_version': package_version({
            'openai':'openai', 'anthropic':'anthropic', 'gemini':'google-genai'
        }[provider]),
    }
    for provider, config in MODEL_CONFIGS.items()
])
display(PROVIDER_ADAPTER_AUDIT)

,provider,model_id,adapter,temperature,reasoning_label,prompt_caching,sdk_version
0,openai,gpt-5.6-terra,call_openai_provider,0.0,reasoning.effort=low,prompt_cache_key with static prefix,2.46.0
1,anthropic,claude-haiku-4-5-20251001,call_anthropic_provider,0.0,lowest setting: extended thinking disabled,top-level cache_control=ephemeral,0.118.0
2,gemini,gemini-3.1-flash-lite,call_gemini_provider,0.0,thinking_level=low,implicit prefix caching; cached token usage logged,2.13.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 13. Isolated provider smoke tests

Run these cells one at a time before enabling a full paid phase. A smoke test submits one synthetic target using P3-D3-C1 and validates the exact schema, model identifier, temperature/reasoning settings, and usage fields. The full runner does not start automatically merely because a smoke test succeeds.

In [55]:
def run_provider_smoke_test(provider: str) -> pd.DataFrame:
    config = MODEL_CONFIGS[provider]
    row = canonicalize_benchmark(synthetic_benchmark()).iloc[2]
    condition = next(c for c in PHASE1_CONDITIONS if c.condition_id == 'P3_D3_ADAPTIVE')
    package = build_prompt_package(row, condition, stage='joint')
    result = dispatch_provider_call(config, package, seed=DISCOVERY_SEED)
    record = {
        'provider': provider,
        'requested_model': result.requested_model,
        'returned_model': result.returned_model,
        'request_id': result.request_id,
        'schema_id': package.schema_id,
        'schema_valid': True,
        'prediction': result.parsed_output.get('unlearning_present'),
        'input_tokens': result.input_tokens,
        'output_tokens': result.output_tokens,
        'cached_input_tokens': result.cached_input_tokens,
        'cache_creation_input_tokens': result.cache_creation_input_tokens,
        'reasoning_tokens': result.reasoning_tokens,
        'latency_seconds': result.latency_seconds,
        'accepted_parameters': canonical_json(result.accepted_parameters),
        'mock': USE_MOCK_PROVIDER,
    }
    path = provider_phase_directory('smoke_tests', provider) / 'smoke_test.json'
    atomic_write_text(path, json.dumps(record, indent=2, default=str))
    return pd.DataFrame([record])

In [56]:
# ISOLATED OPENAI SMOKE TEST
OPENAI_SMOKE = pd.DataFrame()
if USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION):
    try:
        OPENAI_SMOKE = run_provider_smoke_test('openai')
        display(OPENAI_SMOKE)
    except Exception as exc:
        print(f'OpenAI smoke test failed: {type(exc).__name__}: {exc}')
        if RAISE_PROVIDER_CELL_EXCEPTIONS:
            raise
else:
    print('OpenAI smoke test skipped; API execution is disabled.')

OpenAI smoke test failed: BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'prompt_cache_key': string too long. Expected a string with maximum length 64, but got a string with length 86 instead.", 'type': 'invalid_request_error', 'param': 'prompt_cache_key', 'code': 'string_above_max_length'}}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [59]:
# ISOLATED ANTHROPIC SMOKE TEST
ANTHROPIC_SMOKE = pd.DataFrame()
if USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION):
    try:
        ANTHROPIC_SMOKE = run_provider_smoke_test('anthropic')
        display(ANTHROPIC_SMOKE)
    except Exception as exc:
        print(f'Anthropic smoke test failed: {type(exc).__name__}: {exc}')
        if RAISE_PROVIDER_CELL_EXCEPTIONS:
            raise
else:
    print('Anthropic smoke test skipped; API execution is disabled.')

Anthropic smoke test failed: TypeError: Messages.parse() got an unexpected keyword argument 'cache_control'


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [60]:
# ISOLATED GEMINI SMOKE TEST
GEMINI_SMOKE = pd.DataFrame()
if USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION):
    try:
        GEMINI_SMOKE = run_provider_smoke_test('gemini')
        display(GEMINI_SMOKE)
    except Exception as exc:
        print(f'Gemini smoke test failed: {type(exc).__name__}: {exc}')
        if RAISE_PROVIDER_CELL_EXCEPTIONS:
            raise
else:
    print('Gemini smoke test skipped; API execution is disabled.')

,provider,requested_model,returned_model,request_id,schema_id,schema_valid,prediction,input_tokens,output_tokens,cached_input_tokens,cache_creation_input_tokens,reasoning_tokens,latency_seconds,accepted_parameters,mock
0,gemini,gemini-3.1-flash-lite,gemini-3.1-flash-lite,SAVhapbVBrWmmtkPnvq02A8,checklist_joint,True,True,684,245,None,0,100,1.791416,"{""max_output_tokens"":1800,""provider_seed_sent"":false,""response_mime_type"":""application/json"",""temperature"":0.0,""thinking_level"":""low""}",False


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 14. Pricing snapshot and cost estimation

In [61]:
# USD per one million tokens. Snapshot date is part of the manifest because
# provider prices are time-varying. Override through a new protocol version
# rather than retroactively editing a completed run.
PRICING_SNAPSHOT_DATE = '2026-07-22'
TOKEN_PRICING = {
    'openai': {
        'uncached_input': 2.50,
        'cache_write_input': 3.125,
        'cached_input': 0.25,
        'output_including_reasoning': 15.00,
    },
    'anthropic': {
        'uncached_input': 1.00,
        'cache_write_input': 1.25,
        'cached_input': 0.10,
        'output_including_reasoning': 5.00,
    },
    'gemini': {
        'uncached_input': 0.25,
        'cache_write_input': 0.25,
        'cached_input': 0.025,
        'output_including_reasoning': 1.50,
    },
}

def estimate_call_cost_usd(
    provider: str,
    input_tokens: Optional[int],
    output_tokens: Optional[int],
    cached_input_tokens: Optional[int] = 0,
    cache_creation_input_tokens: Optional[int] = 0,
) -> Optional[float]:
    if provider not in TOKEN_PRICING or input_tokens is None or output_tokens is None:
        return None
    rates = TOKEN_PRICING[provider]
    cached = max(0, int(cached_input_tokens or 0))
    created = max(0, int(cache_creation_input_tokens or 0))
    total_input = max(0, int(input_tokens or 0))
    # Provider usage semantics differ. Clamp the residual to avoid negative
    # uncached tokens if a provider reports cache writes outside input_tokens.
    uncached = max(0, total_input - cached - created)
    cost = (
        uncached * rates['uncached_input']
        + created * rates['cache_write_input']
        + cached * rates['cached_input']
        + int(output_tokens or 0) * rates['output_including_reasoning']
    ) / 1_000_000
    return float(cost)

PRICING_MANIFEST = {
    'snapshot_date': PRICING_SNAPSHOT_DATE,
    'currency': 'USD',
    'per_tokens': 1_000_000,
    'rates': TOKEN_PRICING,
    'caution': 'Estimates exclude taxes, storage, priority/flex premiums, and provider billing adjustments.',
}
(CONFIG_ROOT / 'pricing_snapshot.json').write_text(
    json.dumps(PRICING_MANIFEST, indent=2), encoding='utf-8'
)
display(pd.DataFrame(TOKEN_PRICING).T)

,uncached_input,cache_write_input,cached_input,output_including_reasoning
openai,2.50,3.125,0.250,15.0
anthropic,1.00,1.250,0.100,5.0
gemini,0.25,0.250,0.025,1.5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 15. Generic resumable provider runner

In [62]:
def serialize_provider_result(result: ProviderCallResult) -> dict[str, Any]:
    record = asdict(result)
    record['accepted_parameters_json'] = canonical_json(record.pop('accepted_parameters'))
    record['provider_metadata_json'] = canonical_json(record.pop('provider_metadata'))
    record['parsed_output_json'] = canonical_json(record.pop('parsed_output'))
    record['response_sha256'] = sha256_text(record['raw_response_text'])
    record['estimated_cost_usd'] = estimate_call_cost_usd(
        result.provider,
        result.input_tokens,
        result.output_tokens,
        result.cached_input_tokens,
        result.cache_creation_input_tokens,
    )
    return record

def condition_manifest(condition: ConditionSpec) -> dict[str, Any]:
    definition = DEFINITIONS[condition.definition_id]
    return {
        **asdict(condition),
        'definition_name': definition.name,
        'definition_status': definition.status,
        'definition_sha256': sha256_text(definition.operational_text),
    }

def phase_log_paths(phase: str, provider: str) -> dict[str, Path]:
    directory = provider_phase_directory(phase, provider)
    return {
        'directory': directory,
        'requests': directory / 'requests.jsonl',
        'results': directory / 'results.jsonl',
        'errors': directory / 'errors.jsonl',
        'snapshot': directory / 'latest_results.csv',
        'status': directory / 'provider_status.json',
    }

def provider_results_snapshot(path: Path) -> pd.DataFrame:
    latest = list(latest_records_by_run_key(path).values())
    return pd.DataFrame(latest)

def parsed_stage1_lookup_key(
    provider: str,
    condition_id: str,
    row_id: str,
    seed: int,
    repeat_id: str,
) -> tuple[str, str, str, int, str]:
    return provider, condition_id, str(row_id), int(seed), repeat_id

def run_provider_tasks(
    provider: str,
    tasks: Sequence[TaskSpec],
    conditions: Sequence[ConditionSpec],
    stage1_outputs: Optional[dict[tuple[str,str,str,int,str], dict[str,Any]]] = None,
) -> pd.DataFrame:
    if BENCHMARK is None:
        raise RuntimeError(f'Benchmark unavailable: {BENCHMARK_LOAD_ERROR}')
    if provider not in MODEL_CONFIGS:
        raise KeyError(f'Unknown provider: {provider}')
    config = MODEL_CONFIGS[provider]
    condition_map = conditions_by_id(conditions)
    row_map = {str(row.row_id): row for row in BENCHMARK.itertuples(index=False)}
    paths = phase_log_paths(tasks[0].phase if tasks else 'empty', provider)
    success_keys = successful_run_keys(paths['results'])
    new_calls = 0
    stopped_early = False
    stop_reason = None
    started_at = datetime.now(timezone.utc).isoformat()

    for task in tqdm(sorted(tasks, key=lambda item: (item.seed, item.sequence)), desc=f'{provider}:{tasks[0].phase if tasks else "empty"}'):
        condition = condition_map[task.condition_id]
        row_tuple = row_map.get(str(task.row_id))
        if row_tuple is None:
            raise KeyError(f'Row not found: {task.row_id}')
        row = pd.Series(row_tuple._asdict())
        stage1_output = None
        if task.stage == 'target':
            lookup = parsed_stage1_lookup_key(
                provider, task.condition_id, task.row_id, task.seed, task.repeat_id
            )
            stage1_output = (stage1_outputs or {}).get(lookup)
            if not stage1_output:
                # A negative or failed Stage 1 should not generate a Stage-2 request.
                continue
        package = build_prompt_package(row, condition, stage=task.stage, stage1_output=stage1_output)
        run_key = make_run_key(config, task, condition, package)
        if run_key in success_keys:
            continue
        if new_calls >= MAX_NEW_CALLS_PER_PROVIDER_CELL:
            stopped_early = True
            stop_reason = f'MAX_NEW_CALLS_PER_PROVIDER_CELL={MAX_NEW_CALLS_PER_PROVIDER_CELL}'
            break

        request_record = {
            'run_key': run_key,
            'protocol_version': PROTOCOL_VERSION,
            'dataset_sha256': DATASET_SHA256,
            'provider': provider,
            'requested_model': config.model_id,
            'model_config_sha256': model_config_hash(config),
            **asdict(task),
            **condition_manifest(condition),
            'schema_id': package.schema_id,
            'schema_sha256': package.schema_sha256,
            'prompt_sha256': package.prompt_sha256,
            'cache_key': package.cache_key,
            'system_prompt_sha256': sha256_text(package.system_prompt),
            'user_prompt_sha256': sha256_text(package.user_prompt),
            'system_prompt': package.system_prompt,
            'user_prompt': package.user_prompt,
            'output_schema_json': canonical_json(package.schema_model.model_json_schema()),
            'source_blocks_json': canonical_json(package.source_blocks),
            'created_at_utc': datetime.now(timezone.utc).isoformat(),
        }
        append_jsonl(paths['requests'], request_record)
        attempt_started = datetime.now(timezone.utc).isoformat()
        try:
            result = call_with_retry(
                lambda: dispatch_provider_call(config, package, task.seed),
                seed=stable_int_seed(task.seed, provider, task.row_id, task.condition_id),
            )
            # Mandatory local validation, even when provider-native parsing succeeded.
            validated = package.schema_model.model_validate(result.parsed_output).model_dump()
            result.parsed_output = validated
            result_record = {
                **request_record,
                **serialize_provider_result(result),
                'status': 'ok',
                'attempt_started_at_utc': attempt_started,
                'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            append_jsonl(paths['results'], result_record)
            success_keys.add(run_key)
            new_calls += 1
            if REQUEST_SPACING_SECONDS > 0:
                time.sleep(REQUEST_SPACING_SECONDS)
        except Exception as exc:
            classification = error_classification(exc)
            error_record = {
                **request_record,
                'status': 'error',
                'error_classification': classification,
                'error_type': type(exc).__name__,
                'error_message': str(exc),
                'status_code': exception_status_code(exc),
                'traceback': traceback.format_exc(),
                'attempt_started_at_utc': attempt_started,
                'completed_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            append_jsonl(paths['errors'], error_record)
            append_jsonl(paths['results'], error_record)
            new_calls += 1
            if classification == 'configuration' and STOP_PROVIDER_ON_CONFIGURATION_ERROR:
                stopped_early = True
                stop_reason = f'configuration error: {type(exc).__name__}: {exc}'
                break

    snapshot = provider_results_snapshot(paths['results'])
    atomic_write_dataframe_csv(snapshot, paths['snapshot'])
    status = {
        'provider': provider,
        'phase': tasks[0].phase if tasks else None,
        'started_at_utc': started_at,
        'completed_at_utc': datetime.now(timezone.utc).isoformat(),
        'tasks_requested': len(tasks),
        'new_attempts': new_calls,
        'successful_run_keys_total': len(successful_run_keys(paths['results'])),
        'stopped_early': stopped_early,
        'stop_reason': stop_reason,
        'result_log': str(paths['results']),
        'error_log': str(paths['errors']),
    }
    atomic_write_text(paths['status'], json.dumps(status, indent=2, default=str))
    if stopped_early:
        print(f'{provider} stopped early: {stop_reason}')
    return snapshot

def safe_run_provider_cell(
    provider: str,
    tasks: Sequence[TaskSpec],
    conditions: Sequence[ConditionSpec],
    stage1_outputs: Optional[dict[tuple[str,str,str,int,str], dict[str,Any]]] = None,
) -> pd.DataFrame:
    try:
        return run_provider_tasks(provider, tasks, conditions, stage1_outputs)
    except Exception as exc:
        print(f'{provider} provider cell failed without affecting other providers: {type(exc).__name__}: {exc}')
        traceback.print_exc(limit=3)
        if RAISE_PROVIDER_CELL_EXCEPTIONS:
            raise
        return pd.DataFrame()

In [63]:
def load_phase_results(phase: str, providers: Sequence[str] = ('openai','anthropic','gemini')) -> pd.DataFrame:
    frames = []
    for provider in providers:
        path = phase_log_paths(phase, provider)['results']
        frame = provider_results_snapshot(path)
        if not frame.empty:
            frames.append(frame)
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

def parse_output_column(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    data = frame.copy()
    parsed_records = []
    for value in data.get('parsed_output_json', pd.Series([None] * len(data))):
        if isinstance(value, dict):
            parsed_records.append(value)
        elif isinstance(value, str) and value.strip():
            with contextlib.suppress(json.JSONDecodeError):
                parsed_records.append(json.loads(value))
                continue
            parsed_records.append({})
        else:
            parsed_records.append({})
    parsed = pd.json_normalize(parsed_records, sep='__').add_prefix('out__')
    return pd.concat([data.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)

def successful_predictions(phase: str) -> pd.DataFrame:
    frame = load_phase_results(phase)
    if frame.empty:
        return frame
    frame = frame[frame['status'].eq('ok')].copy()
    frame = frame.sort_values('completed_at_utc').drop_duplicates('run_key', keep='last')
    return parse_output_column(frame)

## 16. Evidence validation and protocol-rule audit

In [64]:
def quote_matches_source(quote: Any, source_scope: Any, blocks: dict[str,str]) -> tuple[bool, Optional[str]]:
    quote_text = normalize_for_match(quote)
    scope = normalize_space(source_scope)
    if not quote_text:
        return scope == 'absent', None
    scope_to_key = {
        'target':'target','previous_1':'previous_1','previous_2':'previous_2',
        'next_1':'next_1','next_2':'next_2','metadata':'metadata',
    }
    key = scope_to_key.get(scope)
    if key is None:
        return False, None
    source = normalize_for_match(blocks.get(key, ''))
    if quote_text in source:
        return True, key
    # Exact quote is the rule, but normalized Unicode/whitespace matching avoids
    # false failures caused only by PDF typography.
    return False, key

def extract_evidence_items(parsed: dict[str,Any]) -> list[dict[str,Any]]:
    if isinstance(parsed.get('evidence'), list):
        return [dict(item) for item in parsed['evidence']]
    items = []
    for key in ['prior_state','inadequacy_or_failure','departure_or_reconfiguration']:
        element = parsed.get(key)
        if isinstance(element, dict):
            items.append({
                'element': key,
                'quote': element.get('quote'),
                'source_scope': element.get('source_scope'),
                'identified': element.get('identified'),
            })
    if 'target_evidence_quote' in parsed:
        items.append({
            'element':'other',
            'quote':parsed.get('target_evidence_quote'),
            'source_scope':parsed.get('target_evidence_scope'),
            'identified':bool(parsed.get('target_evidence_quote')),
        })
    return items

def audit_prediction_evidence(record: pd.Series) -> dict[str,Any]:
    if BENCHMARK is None:
        return {}
    row_matches = BENCHMARK[BENCHMARK['row_id'].astype(str).eq(str(record['row_id']))]
    if row_matches.empty:
        return {'evidence_valid':False,'evidence_error':'row not found'}
    row = row_matches.iloc[0]
    condition = ConditionSpec(
        condition_id=record['condition_id'], phase=record['phase'],
        definition_id=record['definition_id'], prompt_style=record['prompt_style'],
        context_id=record['context_id'], workflow=record['workflow'],
        description=record.get('description',''),
    )
    blocks = source_blocks_for_context(row, condition.context_id)
    try:
        parsed = json.loads(record['parsed_output_json'])
    except Exception:
        return {'evidence_valid':False,'schema_valid':False,'evidence_error':'parsed_output_json invalid'}
    schema_id = record['schema_id']
    schema_valid = True
    try:
        SCHEMA_MODELS[schema_id].model_validate(parsed)
    except Exception as exc:
        schema_valid = False
        schema_error = str(exc)
    else:
        schema_error = None

    evidence_items = extract_evidence_items(parsed)
    quote_checks = []
    for item in evidence_items:
        valid, matched_source = quote_matches_source(
            item.get('quote'), item.get('source_scope'), blocks
        )
        quote_checks.append({**item,'valid':valid,'matched_source':matched_source})
    all_quotes_valid = all(item['valid'] for item in quote_checks) if quote_checks else True
    positive = bool(parsed.get('unlearning_present', True if schema_id == 'stage2_target' else False))
    departure_items = [
        item for item in quote_checks
        if item.get('element') == 'departure_or_reconfiguration'
    ]
    target_departure_present = any(
        item.get('valid') and item.get('source_scope') == 'target' and normalize_space(item.get('quote'))
        for item in departure_items
    )
    if schema_id == 'stage2_target':
        target_departure_present = any(
            item.get('valid') and item.get('source_scope') == 'target' and normalize_space(item.get('quote'))
            for item in quote_checks
        )
    no_joint_target_violation = True
    if schema_id in {'simple_joint','checklist_joint'}:
        if positive:
            no_joint_target_violation = parsed.get('target_type') != 'none'
        else:
            no_joint_target_violation = (
                parsed.get('target_type') == 'none' and parsed.get('agency') is None
            )
    d3_additive_violation = (
        record['definition_id'] == 'D3_provisional_adaptive'
        and positive
        and parsed.get('change_type') == 'add_capacity_only'
    )
    checklist_consistent = True
    if schema_id in {'checklist_joint','checklist_binary'}:
        required = ['prior_state','inadequacy_or_failure','departure_or_reconfiguration']
        identified = {name: bool(parsed.get(name,{}).get('identified')) for name in required}
        expected_missing = sorted(name for name, value in identified.items() if not value)
        checklist_consistent = (
            sorted(parsed.get('missing_elements',[])) == expected_missing
            and bool(parsed.get('all_required_elements_present')) == all(identified.values())
        )
    evidence_valid = all([
        schema_valid,
        all_quotes_valid,
        (not positive or target_departure_present),
        no_joint_target_violation,
        not d3_additive_violation,
        checklist_consistent,
    ])
    return {
        'schema_valid': schema_valid,
        'schema_error': schema_error,
        'evidence_quote_count': len(quote_checks),
        'all_quotes_valid': all_quotes_valid,
        'target_departure_present': target_departure_present,
        'joint_target_null_rule_valid': no_joint_target_violation,
        'd3_additive_rule_valid': not d3_additive_violation,
        'checklist_consistent': checklist_consistent,
        'evidence_valid': evidence_valid,
        'evidence_audit_json': canonical_json(quote_checks),
    }

def add_evidence_audit(predictions: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty:
        return predictions.copy()
    audit = predictions.apply(audit_prediction_evidence, axis=1, result_type='expand')
    return pd.concat([predictions.reset_index(drop=True), audit.reset_index(drop=True)], axis=1)

## 17. Gold-label policy and binary evaluation

In [65]:
def complete_gold_column(column: str) -> bool:
    return bool(
        BENCHMARK is not None
        and column in BENCHMARK.columns
        and BENCHMARK[column].notna().all()
    )

def aligned_gold_column(definition_id: str) -> tuple[str,str]:
    preferred = DEFINITION_GOLD_COLUMN.get(definition_id, 'gold_final_definition')
    if complete_gold_column(preferred):
        return preferred, 'definition-aligned adjudicated gold'
    return 'gold_historical', 'historical provisional gold'

def common_final_gold_column() -> tuple[str,str]:
    if complete_gold_column('gold_final_definition'):
        return 'gold_final_definition', 'common final adjudicated gold'
    return 'gold_historical', 'historical provisional gold'

def attach_gold_and_predictions(predictions: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty or BENCHMARK is None:
        return predictions.copy()
    benchmark_columns = [
        'row_id','document_title','gold_historical','gold_old_definition',
        'gold_current_definition','gold_final_definition','gold_target','gold_agency'
    ]
    data = predictions.merge(
        BENCHMARK[benchmark_columns], on='row_id', how='left', validate='many_to_one'
    )
    aligned_values, aligned_basis = [], []
    for _, row in data.iterrows():
        column, basis = aligned_gold_column(row['definition_id'])
        aligned_values.append(row.get(column))
        aligned_basis.append(basis + f' ({column})')
    common_column, common_basis = common_final_gold_column()
    data['gold_definition_aligned'] = aligned_values
    data['gold_definition_aligned_basis'] = aligned_basis
    data['gold_common_final'] = data[common_column]
    data['gold_common_final_basis'] = common_basis + f' ({common_column})'
    data['pred_unlearning'] = data['out__unlearning_present'].map(
        lambda value: 'Yes' if value is True else ('No' if value is False else None)
    )
    data['pred_int'] = data['pred_unlearning'].map(label_to_int)
    data['gold_aligned_int'] = data['gold_definition_aligned'].map(label_to_int)
    data['gold_common_int'] = data['gold_common_final'].map(label_to_int)
    data['confidence'] = pd.to_numeric(data.get('out__confidence'), errors='coerce')
    # Convert confidence in the chosen class to a probability of Yes.
    data['probability_yes'] = np.where(
        data['pred_int'].eq(1), data['confidence'], 1 - data['confidence']
    )
    return data

def safe_rate(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else np.nan

def binary_metric_record(
    group: pd.DataFrame,
    gold_column: str = 'gold_common_int',
) -> dict[str,Any]:
    valid = group.dropna(subset=[gold_column,'pred_int']).copy()
    if valid.empty:
        return {'n_scored':0}
    y_true = valid[gold_column].astype(int).to_numpy()
    y_pred = valid['pred_int'].astype(int).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    probability = pd.to_numeric(valid.get('probability_yes'), errors='coerce')
    brier = float(np.mean((probability - y_true) ** 2)) if probability.notna().all() else np.nan
    return {
        'n_scored':len(valid),
        'human_yes':int(y_true.sum()),
        'human_no':int((1-y_true).sum()),
        'pred_yes':int(y_pred.sum()),
        'pred_no':int((1-y_pred).sum()),
        'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
        'accuracy':accuracy_score(y_true,y_pred),
        'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
        'precision_yes':precision_score(y_true,y_pred,zero_division=0),
        'recall_yes':recall_score(y_true,y_pred,zero_division=0),
        'specificity':safe_rate(tn,tn+fp),
        'f1_yes':f1_score(y_true,y_pred,zero_division=0),
        'mcc':matthews_corrcoef(y_true,y_pred) if len(set(y_true)) > 1 else np.nan,
        'cohen_kappa':cohen_kappa_score(y_true,y_pred) if len(set(y_true)) > 1 else np.nan,
        'npv':safe_rate(tn,tn+fn),
        'brier_score':brier,
    }

def summarize_binary(
    data: pd.DataFrame,
    group_columns: Sequence[str],
    gold_column: str = 'gold_common_int',
) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else list(group_columns)
    for keys, group in data.groupby(grouper, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        row.update(binary_metric_record(group, gold_column))
        row.update({
            'schema_valid_rate':pd.to_numeric(group.get('schema_valid'), errors='coerce').mean(),
            'evidence_valid_rate':pd.to_numeric(group.get('evidence_valid'), errors='coerce').mean(),
            'mean_confidence':pd.to_numeric(group.get('confidence'), errors='coerce').mean(),
            'input_tokens':pd.to_numeric(group.get('input_tokens'), errors='coerce').sum(min_count=1),
            'output_tokens':pd.to_numeric(group.get('output_tokens'), errors='coerce').sum(min_count=1),
            'reasoning_tokens':pd.to_numeric(group.get('reasoning_tokens'), errors='coerce').sum(min_count=1),
            'cached_input_tokens':pd.to_numeric(group.get('cached_input_tokens'), errors='coerce').sum(min_count=1),
            'cache_creation_input_tokens':pd.to_numeric(group.get('cache_creation_input_tokens'), errors='coerce').sum(min_count=1),
            'estimated_cost_usd':pd.to_numeric(group.get('estimated_cost_usd'), errors='coerce').sum(min_count=1),
            'mean_latency_seconds':pd.to_numeric(group.get('latency_seconds'), errors='coerce').mean(),
        })
        rows.append(row)
    return pd.DataFrame(rows)

def prepare_scored_predictions(phase: str) -> pd.DataFrame:
    data = successful_predictions(phase)
    if data.empty:
        return data
    data = add_evidence_audit(data)
    return attach_gold_and_predictions(data)

In [66]:
def condition_ranking(scored: pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame]:
    if scored.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    provider_metrics = summarize_binary(
        scored, ['condition_id','provider','definition_id','prompt_style','context_id','workflow']
    )
    document_metrics = summarize_binary(
        scored, ['condition_id','provider','document_title']
    )
    document_positive = document_metrics[document_metrics['human_yes'].gt(0)].copy()
    worst_document = (
        document_positive.groupby('condition_id')['recall_yes']
        .min().rename('worst_provider_document_recall')
        if not document_positive.empty else pd.Series(dtype=float)
    )
    ranking = (
        provider_metrics.groupby(['condition_id','definition_id','prompt_style','context_id','workflow'], dropna=False)
        .agg(
            providers=('provider','nunique'),
            mean_accuracy=('accuracy','mean'),
            mean_balanced_accuracy=('balanced_accuracy','mean'),
            mean_precision_yes=('precision_yes','mean'),
            mean_recall_yes=('recall_yes','mean'),
            mean_specificity=('specificity','mean'),
            mean_f1_yes=('f1_yes','mean'),
            mean_mcc=('mcc','mean'),
            mean_kappa=('cohen_kappa','mean'),
            schema_valid_rate=('schema_valid_rate','mean'),
            evidence_valid_rate=('evidence_valid_rate','mean'),
            total_cost_usd=('estimated_cost_usd','sum'),
            mean_latency_seconds=('mean_latency_seconds','mean'),
        )
        .reset_index()
        .merge(worst_document, on='condition_id', how='left')
    )
    ranking['eligible'] = (
        ranking['providers'].eq(len(MODEL_CONFIGS))
        & ranking['schema_valid_rate'].ge(MIN_SCHEMA_VALID_RATE)
        & ranking['evidence_valid_rate'].ge(MIN_EVIDENCE_VALID_RATE)
        & ranking['mean_specificity'].ge(SPECIFICITY_FLOOR)
    )
    ranking['constraint_shortfall'] = (
        (len(MODEL_CONFIGS) - ranking['providers']).clip(lower=0)
        + (MIN_SCHEMA_VALID_RATE - ranking['schema_valid_rate']).clip(lower=0)
        + (MIN_EVIDENCE_VALID_RATE - ranking['evidence_valid_rate']).clip(lower=0)
        + (SPECIFICITY_FLOOR - ranking['mean_specificity']).clip(lower=0)
    )
    ranking = ranking.sort_values(
        [
            'eligible','constraint_shortfall','worst_provider_document_recall',
            'mean_f1_yes','mean_mcc','mean_balanced_accuracy',
            'mean_specificity','total_cost_usd','condition_id'
        ],
        ascending=[False,True,False,False,False,False,False,True,True],
        na_position='last',
    ).reset_index(drop=True)
    ranking['rank'] = np.arange(1, len(ranking)+1)
    return ranking, provider_metrics, document_metrics

def select_top_condition_ids(
    ranking: pd.DataFrame,
    n: int,
    override: Sequence[str] = (),
) -> list[str]:
    if override:
        unknown = set(override) - set(ranking['condition_id'])
        if unknown:
            raise ValueError(f'Unknown selection override(s): {sorted(unknown)}')
        return list(override)[:n]
    if ranking.empty:
        return []
    eligible = ranking[ranking['eligible']]
    source = eligible if len(eligible) >= n else ranking
    if eligible.empty and not ALLOW_PROVISIONAL_SELECTION:
        raise RuntimeError('No condition met preregistered constraints and provisional selection is disabled.')
    return source.head(n)['condition_id'].tolist()

### Paired tests and model-based inference

In [67]:
def mcnemar_exact_table(
    scored: pd.DataFrame,
    condition_ids: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    if scored.empty:
        return pd.DataFrame()
    use_conditions = list(condition_ids or sorted(scored['condition_id'].unique()))
    rows = []
    for provider in sorted(scored['provider'].unique()):
        provider_data = scored[scored['provider'].eq(provider)]
        for left_id, right_id in itertools.combinations(use_conditions, 2):
            left = provider_data[provider_data['condition_id'].eq(left_id)][
                ['row_id','seed','gold_common_int','pred_int']
            ].rename(columns={'pred_int':'left_pred'})
            right = provider_data[provider_data['condition_id'].eq(right_id)][
                ['row_id','seed','pred_int']
            ].rename(columns={'pred_int':'right_pred'})
            paired = left.merge(right, on=['row_id','seed'], how='inner').dropna()
            if paired.empty:
                continue
            left_correct = paired['left_pred'].astype(int).eq(paired['gold_common_int'].astype(int))
            right_correct = paired['right_pred'].astype(int).eq(paired['gold_common_int'].astype(int))
            b = int((left_correct & ~right_correct).sum())
            c = int((~left_correct & right_correct).sum())
            discordant = b + c
            p_value = binomtest(min(b,c), discordant, 0.5, alternative='two-sided').pvalue if discordant else 1.0
            rows.append({
                'provider':provider,'condition_left':left_id,'condition_right':right_id,
                'left_only_correct':b,'right_only_correct':c,
                'discordant_pairs':discordant,'mcnemar_exact_p':p_value,
            })
    result = pd.DataFrame(rows)
    if not result.empty:
        if multipletests is not None:
            result['holm_adjusted_p'] = multipletests(
                result['mcnemar_exact_p'], method='holm'
            )[1]
        else:
            result['holm_adjusted_p'] = np.minimum(
                1.0, result['mcnemar_exact_p'] * len(result)
            )
    return result

def fit_correctness_gee(scored: pd.DataFrame, phase: str) -> pd.DataFrame:
    if scored.empty or not RUN_GEE_MODELS or sm is None:
        return pd.DataFrame([{'phase':phase,'status':'skipped'}])
    data = scored.dropna(subset=['gold_common_int','pred_int']).copy()
    data['correct'] = data['gold_common_int'].astype(int).eq(data['pred_int'].astype(int)).astype(int)
    data['cluster_id'] = data['provider'].astype(str) + '::' + data['row_id'].astype(str)
    if data['correct'].nunique() < 2 or data['cluster_id'].nunique() < 5:
        return pd.DataFrame([{'phase':phase,'status':'insufficient variation'}])
    formula = 'correct ~ C(definition_id) + C(prompt_style) + C(context_id) + C(workflow) + C(provider)'
    try:
        model = sm.GEE.from_formula(
            formula, groups='cluster_id', data=data,
            family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()
        )
        fit = model.fit()
        confidence = fit.conf_int()
        return pd.DataFrame({
            'phase':phase,
            'term':fit.params.index,
            'log_odds':fit.params.values,
            'odds_ratio':np.exp(fit.params.values),
            'std_error':fit.bse.values,
            'p_value':fit.pvalues.values,
            'ci_low_or':np.exp(confidence.iloc[:,0].values),
            'ci_high_or':np.exp(confidence.iloc[:,1].values),
            'status':'ok',
        })
    except Exception as exc:
        return pd.DataFrame([{'phase':phase,'status':'failed','error':f'{type(exc).__name__}: {exc}'}])

## 18. Phase 1 — definition × prompt-structure experiment

In [68]:
PHASE1_TASKS: list[TaskSpec] = []
if BENCHMARK is not None and 'phase1' in ACTIVE_PHASES:
    PHASE1_TASKS = make_task_schedule(
        BENCHMARK, PHASE1_CONDITIONS, ACTIVE_SEEDS, phase='phase1', stage='joint'
    )
PHASE1_SCHEDULE = pd.DataFrame([asdict(task) for task in PHASE1_TASKS])
print({
    'phase':'phase1',
    'benchmark_rows':0 if BENCHMARK is None else len(BENCHMARK),
    'conditions':len(PHASE1_CONDITIONS),
    'seeds':ACTIVE_SEEDS,
    'tasks_per_provider':len(PHASE1_TASKS),
    'maximum_requests_all_providers':len(PHASE1_TASKS) * len(MODEL_CONFIGS),
})
display(PHASE1_SCHEDULE.head(15))

{'phase': 'phase1', 'benchmark_rows': 42, 'conditions': 7, 'seeds': (17,), 'tasks_per_provider': 294, 'maximum_requests_all_providers': 882}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,phase,condition_id,row_id,seed,repeat_id,stage,sequence
0,phase1,P1_DIRECT,P042,17,seed_17_r1,joint,0
1,phase1,P3_D3_ADAPTIVE,P042,17,seed_17_r1,joint,1
2,phase1,P3_D2_CURRENT,P042,17,seed_17_r1,joint,2
3,phase1,P2_D3_ADAPTIVE,P042,17,seed_17_r1,joint,3
4,phase1,P3_D1_OLD,P042,17,seed_17_r1,joint,4
5,phase1,P2_D2_CURRENT,P042,17,seed_17_r1,joint,5
6,phase1,P2_D1_OLD,P042,17,seed_17_r1,joint,6
7,phase1,P2_D2_CURRENT,P012,17,seed_17_r1,joint,7
8,phase1,P3_D1_OLD,P012,17,seed_17_r1,joint,8
9,phase1,P2_D3_ADAPTIVE,P012,17,seed_17_r1,joint,9


In [69]:
# PHASE 1 — OPENAI ONLY
PHASE1_OPENAI_RAW = pd.DataFrame()
if PHASE1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE1_OPENAI_RAW = safe_run_provider_cell('openai', PHASE1_TASKS, PHASE1_CONDITIONS)
    display(PHASE1_OPENAI_RAW.tail())
else:
    print('Phase 1 OpenAI execution skipped.')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


openai:phase1:   0%|          | 0/294 [00:00<?, ?it/s]

openai stopped early: configuration error: BadRequestError: Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,run_key,protocol_version,dataset_sha256,provider,requested_model,model_config_sha256,phase,condition_id,row_id,seed,repeat_id,stage,sequence,definition_id,prompt_style,context_id,workflow,description,definition_name,definition_status,definition_sha256,schema_id,schema_sha256,prompt_sha256,cache_key,system_prompt_sha256,user_prompt_sha256,system_prompt,user_prompt,output_schema_json,source_blocks_json,created_at_utc,status,error_classification,error_type,error_message,status_code,traceback,attempt_started_at_utc,completed_at_utc
0,8335a622955282f2fd86119223d47853e85c27c1ec8ef0ccd51ae0f504a14fae,prelangchain_ab_v1,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,openai,gpt-5.6-terra,6a1b619da016ef6ef95e6b667fd1de7a452f2b5ac6c970c501dd3781c4e2a25a,phase1,P1_DIRECT,P042,17,seed_17_r1,joint,0,D0_none,P1_direct,C1_target,W1_one_stage,Direct task; no supplied definition.,No supplied definition,control,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855,simple_joint,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e,18e60d865159bd38aad5fb149e1e5380fa6e3f4f2cae3a3aff3bed600110d4a3,prelangchain_ab_v1:D0_none:P1_direct:joint:simple_joint,c84220485aa7c03d97db321902bb1b8e03bb52823a594243a7a8efdd65585d07,a9c5b6d5ad4a3e442a7ecec52f18e3a8fdc85de2cdee3ecd654c8144b55db8cd,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,ROW ID: P042\n\n<TARGET>\nKey Recommendations: CONGRESS SHOULD • ...give the president authority to create a new Citizen Preparedness Directorate. • ...give the president greater authority to strengthen the federal g...,"{""$defs"":{""EvidenceQuote"":{""additionalProperties"":false,""properties"":{""element"":{""enum"":[""prior_state"",""inadequacy_or_failure"",""departure_or_reconfiguration"",""other""],""title"":""Element"",""type"":""string""},""quote"":{""anyO...","{""target"":""Key Recommendations: CONGRESS SHOULD • ...give the president authority to create a new Citizen Preparedness Directorate. • ...give the president greater authority to strengthen the federal government’s hum...",2026-07-22T18:02:07.299962+00:00,error,configuration,BadRequestError,"Error code: 400 - {'error': {'message': ""Unsupported parameter: 'temperature' is not supported with this model."", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}",400,"Traceback (most recent call last):\n File ""/tmp/ipykernel_1882/3291723974.py"", line 117, in run_provider_tasks\n result = call_with_retry(\n ^^^^^^^^^^^^^^^^\n File ""/tmp/ipykernel_1882/2099912802.py...",2026-07-22T18:02:07.300460+00:00,2026-07-22T18:02:07.609535+00:00


In [70]:
# PHASE 1 — ANTHROPIC ONLY
PHASE1_ANTHROPIC_RAW = pd.DataFrame()
if PHASE1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE1_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE1_TASKS, PHASE1_CONDITIONS)
    display(PHASE1_ANTHROPIC_RAW.tail())
else:
    print('Phase 1 Anthropic execution skipped.')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


anthropic:phase1:   0%|          | 0/294 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,run_key,protocol_version,dataset_sha256,provider,requested_model,model_config_sha256,phase,condition_id,row_id,seed,repeat_id,stage,sequence,definition_id,prompt_style,context_id,workflow,description,definition_name,definition_status,definition_sha256,schema_id,schema_sha256,prompt_sha256,cache_key,system_prompt_sha256,user_prompt_sha256,system_prompt,user_prompt,output_schema_json,source_blocks_json,created_at_utc,status,error_classification,error_type,error_message,status_code,traceback,attempt_started_at_utc,completed_at_utc
289,18f461b5838508ad6e12dcbf9e1e2cf80a6322ae5fe4cdfc0daf5d280309231e,prelangchain_ab_v1,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,anthropic,claude-haiku-4-5-20251001,86832fd651e7d1d4efead0530bec98029d5d913a0d7f0252f5a429f982aeca91,phase1,P3_D3_ADAPTIVE,P034,17,seed_17_r1,joint,289,D3_provisional_adaptive,P3_evidence_checklist,C1_target,W1_one_stage,Adaptive candidate + explicit evidence checklist.,Provisional adaptive-reconfiguration definition,exploratory candidate; requires later human adjudication,99af0841b61738d090c0ecccff0f9c77d35f487944d743c52fa3dc11fcf595d5,checklist_joint,160ef51b848bbba94ee5996a63b20821792de7b7319a848da6741e11e321f8b6,ce21a2da0d447054dd51fed416be4a3ae4fa3225ffcde17a5aec6c94df4ce99c,prelangchain_ab_v1:D3_provisional_adaptive:P3_evidence_checklist:joint:checklist_joint,68d8c7e2563cd42e71b50336e7baa14501ee163ce0137d779de0d05b3332120c,b595969f94646b54d6f1acb26586463fb14ad6d8090d869c97b4cabd338ff122,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,"ROW ID: P034\n\n<TARGET>\nSecretary Chertoff has announced plans to emphasize several of these capabilities in the near term. For example, DHS will acquire a hardened set of communications capabilities, including equ...","{""$defs"":{""ChecklistElement"":{""additionalProperties"":false,""properties"":{""identified"":{""title"":""Identified"",""type"":""boolean""},""quote"":{""anyOf"":[{""type"":""string""},{""type"":""null""}],""description"":""Exact short excerpt fr...","{""target"":""Secretary Chertoff has announced plans to emphasize several of these capabilities in the near term. For example, DHS will acquire a hardened set of communications capabilities, including equipment and spec...",2026-07-22T18:02:24.502282+00:00,error,unknown,TypeError,Messages.parse() got an unexpected keyword argument 'cache_control',None,"Traceback (most recent call last):\n File ""/tmp/ipykernel_1882/3291723974.py"", line 117, in run_provider_tasks\n result = call_with_retry(\n ^^^^^^^^^^^^^^^^\n File ""/tmp/ipykernel_1882/2099912802.py...",2026-07-22T18:02:24.502526+00:00,2026-07-22T18:02:24.503176+00:00
290,c22ab60be16fc87554aa24923e938c844fa4d19cd731668475789ff5a6b64436,prelangchain_ab_v1,ab203fea011cbf56eb75efcb5a2448c713b7640dc8f1deedfdfe2e9673f34ea0,anthropic,claude-haiku-4-5-20251001,86832fd651e7d1d4efead0530bec98029d5d913a0d7f0252f5a429f982aeca91,phase1,P2_D1_OLD,P034,17,seed_17_r1,joint,290,D1_old_broad,P2_simple_definition,C1_target,W1_one_stage,Old definition + simple structured classification.,Old broad operational definition,historical treatment,7531f2c94a782ca6a2642e01fca30d3bd631f94fb2a4e8fad4a90e82dfad513e,simple_joint,b6fae6ca706adbcd29328a1015fe3b7e1788e2e54977bfe56b5b790981eb481e,e9d40b7b0ebe763e8d4a9b44f5bbc42e9a88cc76e785033ccb92d77e530ec01e,prelangchain_ab_v1:D1_old_broad:P2_simple_definition:joint:simple_joint,4f27d500cd74db97ad3b40ef8f4fd2b12172fd96af63d74b86d0ce575fda9712,b595969f94646b54d6f1acb26586463fb14ad6d8090d869c97b4cabd338ff122,You are an independent qualitative-coding model for a reproducible benchmark on organizational unlearning in government agencies.\n\nClassify one TARGET paragraph per request. Never infer or reproduce a human label. ...,"ROW ID: P034\n\n<TARGET>\nSecretary Chertoff has announced plans to emphasize several of 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# PHASE 1 — GEMINI ONLY
PHASE1_GEMINI_RAW = pd.DataFrame()
if PHASE1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE1_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE1_TASKS, PHASE1_CONDITIONS)
    display(PHASE1_GEMINI_RAW.tail())
else:
    print('Phase 1 Gemini execution skipped.')

### Phase 1 analysis and selection

In [ ]:
PHASE1_SCORED = prepare_scored_predictions('phase1')
PHASE1_RANKING, PHASE1_PROVIDER_METRICS, PHASE1_DOCUMENT_METRICS = condition_ranking(PHASE1_SCORED)
PHASE1_MCNEMAR = mcnemar_exact_table(PHASE1_SCORED)
PHASE1_GEE = fit_correctness_gee(PHASE1_SCORED, 'phase1')

if not PHASE1_RANKING.empty:
    display(PHASE1_RANKING)
    display(PHASE1_PROVIDER_METRICS.sort_values(['condition_id','provider']))
    display(PHASE1_DOCUMENT_METRICS.sort_values(['condition_id','provider','document_title']))
else:
    print('No Phase 1 successful predictions available yet.')

In [ ]:
# Definition-aligned and common-final comparisons are shown side by side.
PHASE1_ALIGNED_METRICS = pd.DataFrame()
PHASE1_DEFINITION_TRADEOFF = pd.DataFrame()
PHASE1_EPA_DIAGNOSTICS = pd.DataFrame()
if not PHASE1_SCORED.empty:
    PHASE1_ALIGNED_METRICS = summarize_binary(
        PHASE1_SCORED,
        ['condition_id','provider','definition_id','prompt_style'],
        gold_column='gold_aligned_int',
    )
    common = summarize_binary(
        PHASE1_SCORED,
        ['condition_id','provider','definition_id','prompt_style'],
        gold_column='gold_common_int',
    )
    keep = ['condition_id','provider','definition_id','prompt_style','accuracy','recall_yes','specificity','f1_yes','mcc']
    PHASE1_DEFINITION_TRADEOFF = PHASE1_ALIGNED_METRICS[keep].merge(
        common[keep], on=['condition_id','provider','definition_id','prompt_style'],
        suffixes=('_aligned','_common_final')
    )
    PHASE1_EPA_DIAGNOSTICS = summarize_binary(
        PHASE1_SCORED[
            PHASE1_SCORED['document_title'].str.contains('EPA', case=False, na=False)
        ],
        ['condition_id','provider','definition_id','prompt_style'],
        gold_column='gold_common_int',
    )
    display(PHASE1_DEFINITION_TRADEOFF)
    display(PHASE1_EPA_DIAGNOSTICS)

In [ ]:
PHASE1_SELECTED_IDS = select_top_condition_ids(
    PHASE1_RANKING, n=2, override=PHASE1_SELECTION_OVERRIDE
) if not PHASE1_RANKING.empty else list(PHASE1_SELECTION_OVERRIDE)
PHASE1_SELECTION = {
    'selected_condition_ids': PHASE1_SELECTED_IDS,
    'selection_basis': (
        'override' if PHASE1_SELECTION_OVERRIDE
        else 'preregistered constrained ranking against common-final gold'
    ),
    'provisional_gold': not complete_gold_column('gold_final_definition'),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'phase1_ranking_sha256': sha256_text(
        PHASE1_RANKING.fillna('').to_csv(index=False) if not PHASE1_RANKING.empty else ''
    ),
}
atomic_write_text(
    CONFIG_ROOT / 'phase1_selection.json',
    json.dumps(PHASE1_SELECTION, indent=2, default=str)
)
print(PHASE1_SELECTION)

## 19. Phase 2 — matched context A/B test

In [ ]:
def load_selection_json(name: str) -> dict[str,Any]:
    path = CONFIG_ROOT / name
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}

def resolve_phase1_selected_ids() -> list[str]:
    if PHASE1_SELECTION_OVERRIDE:
        return list(PHASE1_SELECTION_OVERRIDE)
    if 'PHASE1_SELECTED_IDS' in globals() and PHASE1_SELECTED_IDS:
        return list(PHASE1_SELECTED_IDS)
    return list(load_selection_json('phase1_selection.json').get('selected_condition_ids', []))

def phase2_conditions_from_phase1(selected_ids: Sequence[str]) -> list[ConditionSpec]:
    base_map = conditions_by_id(PHASE1_CONDITIONS)
    contexts: list[ContextLevel] = [
        'C1_target','C2_metadata','C3_plusminus1','C4_plusminus2'
    ]
    conditions = []
    for base_id in selected_ids:
        if base_id not in base_map:
            raise ValueError(f'Phase 1 selection not registered: {base_id}')
        base = base_map[base_id]
        for context_id in contexts:
            conditions.append(ConditionSpec(
                condition_id=f'{base_id}__{context_id}',
                phase='phase2', definition_id=base.definition_id,
                prompt_style=base.prompt_style, context_id=context_id,
                workflow='W1_one_stage',
                description=f'{base.description} Context treatment {context_id}.',
            ))
    return conditions

PHASE1_SELECTED_FOR_CONTEXT = resolve_phase1_selected_ids()
PHASE2_CONDITIONS = phase2_conditions_from_phase1(PHASE1_SELECTED_FOR_CONTEXT) if PHASE1_SELECTED_FOR_CONTEXT else []
# C1 is reused from Phase 1 to avoid paying for an identical prompt. Only C2–C4
# generate new provider calls.
PHASE2_NEW_CONDITIONS = [c for c in PHASE2_CONDITIONS if c.context_id != 'C1_target']
PHASE2_TASKS: list[TaskSpec] = []
if BENCHMARK is not None and PHASE2_NEW_CONDITIONS and 'phase2' in ACTIVE_PHASES:
    PHASE2_TASKS = make_task_schedule(
        BENCHMARK, PHASE2_NEW_CONDITIONS, ACTIVE_SEEDS, phase='phase2', stage='joint'
    )

display(pd.DataFrame([asdict(c) for c in PHASE2_CONDITIONS]))
print({
    'selected_phase1':PHASE1_SELECTED_FOR_CONTEXT,
    'new_tasks_per_provider':len(PHASE2_TASKS),
    'C1_reused':True,
})

In [ ]:
# PHASE 2 — OPENAI ONLY
PHASE2_OPENAI_RAW = pd.DataFrame()
if PHASE2_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE2_OPENAI_RAW = safe_run_provider_cell('openai', PHASE2_TASKS, PHASE2_NEW_CONDITIONS)
    display(PHASE2_OPENAI_RAW.tail())
else:
    print('Phase 2 OpenAI execution skipped.')

In [ ]:
# PHASE 2 — ANTHROPIC ONLY
PHASE2_ANTHROPIC_RAW = pd.DataFrame()
if PHASE2_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE2_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE2_TASKS, PHASE2_NEW_CONDITIONS)
    display(PHASE2_ANTHROPIC_RAW.tail())
else:
    print('Phase 2 Anthropic execution skipped.')

In [ ]:
# PHASE 2 — GEMINI ONLY
PHASE2_GEMINI_RAW = pd.DataFrame()
if PHASE2_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE2_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE2_TASKS, PHASE2_NEW_CONDITIONS)
    display(PHASE2_GEMINI_RAW.tail())
else:
    print('Phase 2 Gemini execution skipped.')

### Phase 2 assembly, analysis, and selection

In [ ]:
def reused_phase1_c1_rows() -> pd.DataFrame:
    if 'PHASE1_SCORED' not in globals() or PHASE1_SCORED.empty:
        return pd.DataFrame()
    mapping = {
        base_id: f'{base_id}__C1_target'
        for base_id in PHASE1_SELECTED_FOR_CONTEXT
    }
    rows = PHASE1_SCORED[PHASE1_SCORED['condition_id'].isin(mapping)].copy()
    if rows.empty:
        return rows
    rows['source_phase'] = 'phase1'
    rows['phase'] = 'phase2'
    rows['condition_id'] = rows['condition_id'].map(mapping)
    rows['context_id'] = 'C1_target'
    rows['reused_from_phase1'] = True
    return rows

def prepare_phase2_scored() -> pd.DataFrame:
    new = prepare_scored_predictions('phase2')
    if not new.empty:
        new['reused_from_phase1'] = False
        new['source_phase'] = 'phase2'
    reused = reused_phase1_c1_rows()
    frames = [frame for frame in [reused,new] if not frame.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

PHASE2_SCORED = prepare_phase2_scored()
PHASE2_RANKING, PHASE2_PROVIDER_METRICS, PHASE2_DOCUMENT_METRICS = condition_ranking(PHASE2_SCORED)
PHASE2_MCNEMAR = mcnemar_exact_table(PHASE2_SCORED)
PHASE2_GEE = fit_correctness_gee(PHASE2_SCORED, 'phase2')
if not PHASE2_RANKING.empty:
    display(PHASE2_RANKING)
    display(PHASE2_DOCUMENT_METRICS.sort_values(['condition_id','provider','document_title']))
else:
    print('No Phase 2 successful predictions available yet.')

In [ ]:
if PHASE2_SELECTION_OVERRIDE:
    if PHASE2_SELECTION_OVERRIDE not in set(PHASE2_RANKING.get('condition_id', [])):
        raise ValueError(f'Unknown PHASE2_SELECTION_OVERRIDE={PHASE2_SELECTION_OVERRIDE}')
    PHASE2_SELECTED_ID = PHASE2_SELECTION_OVERRIDE
else:
    selected = select_top_condition_ids(PHASE2_RANKING, n=1)
    PHASE2_SELECTED_ID = selected[0] if selected else ''
PHASE2_SELECTION = {
    'selected_condition_id':PHASE2_SELECTED_ID,
    'selection_basis':'override' if PHASE2_SELECTION_OVERRIDE else 'preregistered constrained ranking',
    'provisional_gold':not complete_gold_column('gold_final_definition'),
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
}
atomic_write_text(
    CONFIG_ROOT / 'phase2_selection.json',
    json.dumps(PHASE2_SELECTION, indent=2, default=str)
)
print(PHASE2_SELECTION)

## 20. Phase 3 — one-stage joint vs binary-first hierarchical workflow

In [ ]:
def resolve_phase2_selected_id() -> str:
    if PHASE2_SELECTION_OVERRIDE:
        return PHASE2_SELECTION_OVERRIDE
    if 'PHASE2_SELECTED_ID' in globals() and PHASE2_SELECTED_ID:
        return PHASE2_SELECTED_ID
    return str(load_selection_json('phase2_selection.json').get('selected_condition_id',''))

def selected_phase2_condition() -> Optional[ConditionSpec]:
    selected_id = resolve_phase2_selected_id()
    mapping = conditions_by_id(PHASE2_CONDITIONS) if PHASE2_CONDITIONS else {}
    return mapping.get(selected_id)

PHASE2_WINNER = selected_phase2_condition()
PHASE3_CONDITIONS: list[ConditionSpec] = []
if PHASE2_WINNER is not None:
    common = dict(
        phase='phase3', definition_id=PHASE2_WINNER.definition_id,
        prompt_style=PHASE2_WINNER.prompt_style,
        context_id=PHASE2_WINNER.context_id,
    )
    PHASE3_CONDITIONS = [
        ConditionSpec(
            condition_id='W1_ONE_STAGE', workflow='W1_one_stage',
            description='One request jointly returns binary decision, structured evidence, target, and agency.',
            **common,
        ),
        ConditionSpec(
            condition_id='W2_BINARY_FIRST', workflow='W2_binary_first',
            description='Stage 1 returns binary/evidence; Stage 2 assigns target/agency only after a positive.',
            **common,
        ),
    ]
PHASE3_W1 = [c for c in PHASE3_CONDITIONS if c.workflow == 'W1_one_stage']
PHASE3_W2 = [c for c in PHASE3_CONDITIONS if c.workflow == 'W2_binary_first']
PHASE3_W1_TASKS = make_task_schedule(BENCHMARK, PHASE3_W1, ACTIVE_SEEDS, 'phase3', 'joint') if BENCHMARK is not None and PHASE3_W1 and 'phase3' in ACTIVE_PHASES else []
PHASE3_W2_BINARY_TASKS = make_task_schedule(BENCHMARK, PHASE3_W2, ACTIVE_SEEDS, 'phase3', 'binary') if BENCHMARK is not None and PHASE3_W2 and 'phase3' in ACTIVE_PHASES else []
PHASE3_W2_TARGET_TASKS = make_task_schedule(BENCHMARK, PHASE3_W2, ACTIVE_SEEDS, 'phase3', 'target') if BENCHMARK is not None and PHASE3_W2 and 'phase3' in ACTIVE_PHASES else []
display(pd.DataFrame([asdict(c) for c in PHASE3_CONDITIONS]))
print({
    'phase2_winner':None if PHASE2_WINNER is None else PHASE2_WINNER.condition_id,
    'W1_tasks_per_provider':len(PHASE3_W1_TASKS),
    'W2_stage1_tasks_per_provider':len(PHASE3_W2_BINARY_TASKS),
    'W2_stage2_ceiling_per_provider':len(PHASE3_W2_TARGET_TASKS),
})

### Phase 3A — isolated one-stage provider cells

In [ ]:
# PHASE 3 W1 — OPENAI ONLY
PHASE3_W1_OPENAI_RAW = pd.DataFrame()
if PHASE3_W1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W1_OPENAI_RAW = safe_run_provider_cell('openai', PHASE3_W1_TASKS, PHASE3_W1)
    display(PHASE3_W1_OPENAI_RAW.tail())
else:
    print('Phase 3 W1 OpenAI execution skipped.')

In [ ]:
# PHASE 3 W1 — ANTHROPIC ONLY
PHASE3_W1_ANTHROPIC_RAW = pd.DataFrame()
if PHASE3_W1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W1_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE3_W1_TASKS, PHASE3_W1)
    display(PHASE3_W1_ANTHROPIC_RAW.tail())
else:
    print('Phase 3 W1 Anthropic execution skipped.')

In [ ]:
# PHASE 3 W1 — GEMINI ONLY
PHASE3_W1_GEMINI_RAW = pd.DataFrame()
if PHASE3_W1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W1_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE3_W1_TASKS, PHASE3_W1)
    display(PHASE3_W1_GEMINI_RAW.tail())
else:
    print('Phase 3 W1 Gemini execution skipped.')

### Phase 3B — isolated binary-first Stage-1 provider cells

In [ ]:
# PHASE 3 W2 STAGE 1 — OPENAI ONLY
PHASE3_W2_BINARY_OPENAI_RAW = pd.DataFrame()
if PHASE3_W2_BINARY_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W2_BINARY_OPENAI_RAW = safe_run_provider_cell('openai', PHASE3_W2_BINARY_TASKS, PHASE3_W2)
    display(PHASE3_W2_BINARY_OPENAI_RAW.tail())
else:
    print('Phase 3 W2 Stage 1 OpenAI execution skipped.')

In [ ]:
# PHASE 3 W2 STAGE 1 — ANTHROPIC ONLY
PHASE3_W2_BINARY_ANTHROPIC_RAW = pd.DataFrame()
if PHASE3_W2_BINARY_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W2_BINARY_ANTHROPIC_RAW = safe_run_provider_cell('anthropic', PHASE3_W2_BINARY_TASKS, PHASE3_W2)
    display(PHASE3_W2_BINARY_ANTHROPIC_RAW.tail())
else:
    print('Phase 3 W2 Stage 1 Anthropic execution skipped.')

In [ ]:
# PHASE 3 W2 STAGE 1 — GEMINI ONLY
PHASE3_W2_BINARY_GEMINI_RAW = pd.DataFrame()
if PHASE3_W2_BINARY_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W2_BINARY_GEMINI_RAW = safe_run_provider_cell('gemini', PHASE3_W2_BINARY_TASKS, PHASE3_W2)
    display(PHASE3_W2_BINARY_GEMINI_RAW.tail())
else:
    print('Phase 3 W2 Stage 1 Gemini execution skipped.')

In [ ]:
def positive_stage1_lookup(phase: str, provider: str, condition_id: str) -> dict[tuple[str,str,str,int,str],dict[str,Any]]:
    predictions = successful_predictions(phase)
    if predictions.empty:
        return {}
    subset = predictions[
        predictions['provider'].eq(provider)
        & predictions['condition_id'].eq(condition_id)
        & predictions['stage'].eq('binary')
        & predictions['out__unlearning_present'].eq(True)
    ]
    lookup = {}
    for _, record in subset.iterrows():
        lookup[parsed_stage1_lookup_key(
            provider, condition_id, record['row_id'], int(record['seed']), record['repeat_id']
        )] = json.loads(record['parsed_output_json'])
    return lookup

PHASE3_STAGE1_LOOKUPS = {
    provider: positive_stage1_lookup('phase3', provider, 'W2_BINARY_FIRST')
    for provider in MODEL_CONFIGS
}
print({provider:len(lookup) for provider,lookup in PHASE3_STAGE1_LOOKUPS.items()})

### Phase 3C — isolated Stage-2 target/agency provider cells

In [ ]:
# PHASE 3 W2 STAGE 2 — OPENAI ONLY
PHASE3_W2_TARGET_OPENAI_RAW = pd.DataFrame()
if PHASE3_W2_TARGET_TASKS and PHASE3_STAGE1_LOOKUPS.get('openai') and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W2_TARGET_OPENAI_RAW = safe_run_provider_cell(
        'openai', PHASE3_W2_TARGET_TASKS, PHASE3_W2, PHASE3_STAGE1_LOOKUPS['openai']
    )
    display(PHASE3_W2_TARGET_OPENAI_RAW.tail())
else:
    print('Phase 3 W2 Stage 2 OpenAI execution skipped or no positive Stage-1 rows.')

In [ ]:
# PHASE 3 W2 STAGE 2 — ANTHROPIC ONLY
PHASE3_W2_TARGET_ANTHROPIC_RAW = pd.DataFrame()
if PHASE3_W2_TARGET_TASKS and PHASE3_STAGE1_LOOKUPS.get('anthropic') and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W2_TARGET_ANTHROPIC_RAW = safe_run_provider_cell(
        'anthropic', PHASE3_W2_TARGET_TASKS, PHASE3_W2, PHASE3_STAGE1_LOOKUPS['anthropic']
    )
    display(PHASE3_W2_TARGET_ANTHROPIC_RAW.tail())
else:
    print('Phase 3 W2 Stage 2 Anthropic execution skipped or no positive Stage-1 rows.')

In [ ]:
# PHASE 3 W2 STAGE 2 — GEMINI ONLY
PHASE3_W2_TARGET_GEMINI_RAW = pd.DataFrame()
if PHASE3_W2_TARGET_TASKS and PHASE3_STAGE1_LOOKUPS.get('gemini') and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    PHASE3_W2_TARGET_GEMINI_RAW = safe_run_provider_cell(
        'gemini', PHASE3_W2_TARGET_TASKS, PHASE3_W2, PHASE3_STAGE1_LOOKUPS['gemini']
    )
    display(PHASE3_W2_TARGET_GEMINI_RAW.tail())
else:
    print('Phase 3 W2 Stage 2 Gemini execution skipped or no positive Stage-1 rows.')

### Phase 3 workflow assembly and evaluation

In [ ]:
MERGE_KEYS = ['provider','condition_id','row_id','seed','repeat_id']
USAGE_COLUMNS = [
    'input_tokens','output_tokens','total_tokens','reasoning_tokens',
    'cached_input_tokens','cache_creation_input_tokens','estimated_cost_usd','latency_seconds'
]

def assemble_workflow_predictions(phase: str) -> pd.DataFrame:
    raw = successful_predictions(phase)
    if raw.empty:
        return raw
    joint = raw[raw['stage'].eq('joint')].copy()
    binary = raw[raw['stage'].eq('binary')].copy()
    target = raw[raw['stage'].eq('target')].copy()
    if not joint.empty:
        joint['workflow_complete'] = True
        joint['stage2_called'] = False
    if not binary.empty:
        target_fields = MERGE_KEYS + [
            'out__target_type','out__agency','out__target_evidence_quote',
            'out__target_evidence_scope','out__confidence','out__needs_human_review'
        ] + [column for column in USAGE_COLUMNS if column in target.columns]
        target_keep = target[[c for c in target_fields if c in target.columns]].copy()
        rename = {
            'out__target_type':'stage2__target_type',
            'out__agency':'stage2__agency',
            'out__target_evidence_quote':'stage2__target_evidence_quote',
            'out__target_evidence_scope':'stage2__target_evidence_scope',
            'out__confidence':'stage2__confidence',
            'out__needs_human_review':'stage2__needs_human_review',
            **{column:f'stage2__{column}' for column in USAGE_COLUMNS if column in target_keep.columns},
        }
        target_keep = target_keep.rename(columns=rename)
        binary = binary.merge(target_keep, on=MERGE_KEYS, how='left', validate='one_to_one')
        positive = binary['out__unlearning_present'].eq(True)
        binary['out__target_type'] = np.where(
            positive, binary.get('stage2__target_type'), 'none'
        )
        binary['out__agency'] = np.where(
            positive, binary.get('stage2__agency'), None
        )
        binary['stage2_called'] = positive & binary.get('stage2__target_type', pd.Series(index=binary.index,dtype=object)).notna()
        binary['workflow_complete'] = (~positive) | binary['stage2_called']
        for column in USAGE_COLUMNS:
            left = pd.to_numeric(binary.get(column), errors='coerce').fillna(0)
            right = pd.to_numeric(binary.get(f'stage2__{column}'), errors='coerce').fillna(0)
            binary[column] = left + right
    frames = [frame for frame in [joint,binary] if not frame.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

PHASE3_UNIFIED = assemble_workflow_predictions('phase3')
PHASE3_SCORED = attach_gold_and_predictions(add_evidence_audit(PHASE3_UNIFIED)) if not PHASE3_UNIFIED.empty else pd.DataFrame()
PHASE3_RANKING, PHASE3_PROVIDER_METRICS, PHASE3_DOCUMENT_METRICS = condition_ranking(PHASE3_SCORED)
PHASE3_MCNEMAR = mcnemar_exact_table(PHASE3_SCORED)
PHASE3_GEE = fit_correctness_gee(PHASE3_SCORED, 'phase3')
if not PHASE3_RANKING.empty:
    display(PHASE3_RANKING)
    display(PHASE3_PROVIDER_METRICS)
else:
    print('No Phase 3 successful workflow predictions available yet.')

In [ ]:
TARGET_NORMALIZATION = {
    'leadership':'leadership',
    'laws plans and policies':'laws_plans_policies',
    'laws plans policies':'laws_plans_policies',
    'laws_plans_policies':'laws_plans_policies',
    'capabilities':'capabilities',
    'funds and resources':'funds_resources',
    'funds resources':'funds_resources',
    'funds_resources':'funds_resources',
    'misc organizational':'misc_organizational',
    'miscellaneous organizational':'misc_organizational',
    'misc_organizational':'misc_organizational',
    'none':'none',
}

def normalize_target(value: Any) -> Optional[str]:
    text = normalize_for_match(value).replace('/',' ').replace('_',' ')
    text = re.sub(r'[^a-z0-9 ]+',' ',text)
    text = normalize_space(text)
    return TARGET_NORMALIZATION.get(text, text.replace(' ','_') if text else None)

def agency_token_set(value: Any) -> set[str]:
    return {
        token for token in re.findall(r'[a-z0-9]+', normalize_for_match(value))
        if token not in {'the','of','and','department','office','agency','government'}
    }

def target_agency_metrics(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    scored = data.copy()
    scored['gold_target_norm'] = scored['gold_target'].map(normalize_target)
    scored['pred_target_norm'] = scored['out__target_type'].map(normalize_target)
    gold_positive = scored['gold_common_int'].eq(1)
    scored['target_exact'] = np.where(
        gold_positive,
        scored['gold_target_norm'].eq(scored['pred_target_norm']),
        np.nan,
    )
    scored['binary_target_exact'] = (
        scored['gold_common_int'].eq(scored['pred_int'])
        & np.where(gold_positive, scored['gold_target_norm'].eq(scored['pred_target_norm']), True)
    )
    scored['agency_overlap'] = scored.apply(
        lambda row: (
            np.nan if not bool(row['gold_common_int']) or not normalize_space(row.get('gold_agency'))
            else bool(agency_token_set(row.get('gold_agency')) & agency_token_set(row.get('out__agency')))
        ), axis=1
    )
    return (
        scored.groupby(['condition_id','provider','workflow'], dropna=False)
        .agg(
            rows=('row_id','count'),
            target_accuracy_on_gold_positive=('target_exact','mean'),
            binary_plus_target_exact=('binary_target_exact','mean'),
            agency_overlap_accuracy=('agency_overlap','mean'),
            workflow_complete_rate=('workflow_complete','mean'),
            stage2_call_rate=('stage2_called','mean'),
        )
        .reset_index()
    )

PHASE3_TARGET_METRICS = target_agency_metrics(PHASE3_SCORED)
display(PHASE3_TARGET_METRICS)

In [ ]:
if PHASE3_WORKFLOW_OVERRIDE:
    workflow_matches = PHASE3_RANKING[
        PHASE3_RANKING['workflow'].eq(PHASE3_WORKFLOW_OVERRIDE)
    ]
    if workflow_matches.empty:
        raise ValueError(f'Unknown PHASE3_WORKFLOW_OVERRIDE={PHASE3_WORKFLOW_OVERRIDE}')
    PHASE3_SELECTED_ID = workflow_matches.iloc[0]['condition_id']
else:
    selected = select_top_condition_ids(PHASE3_RANKING, n=1)
    PHASE3_SELECTED_ID = selected[0] if selected else ''
PHASE3_SELECTED_CONDITION = conditions_by_id(PHASE3_CONDITIONS).get(PHASE3_SELECTED_ID) if PHASE3_CONDITIONS else None
PHASE3_SELECTION = {
    'selected_condition_id':PHASE3_SELECTED_ID,
    'selected_workflow':None if PHASE3_SELECTED_CONDITION is None else PHASE3_SELECTED_CONDITION.workflow,
    'selection_basis':'override' if PHASE3_WORKFLOW_OVERRIDE else 'preregistered constrained ranking',
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
}
atomic_write_text(
    CONFIG_ROOT / 'phase3_selection.json',
    json.dumps(PHASE3_SELECTION, indent=2, default=str)
)
print(PHASE3_SELECTION)

## 21. Finalist stability experiment across multiple seeds

In [ ]:
def resolve_phase3_selected_condition() -> Optional[ConditionSpec]:
    if 'PHASE3_SELECTED_CONDITION' in globals() and PHASE3_SELECTED_CONDITION is not None:
        return PHASE3_SELECTED_CONDITION
    saved = load_selection_json('phase3_selection.json')
    selected_id = saved.get('selected_condition_id','')
    mapping = conditions_by_id(PHASE3_CONDITIONS) if PHASE3_CONDITIONS else {}
    return mapping.get(selected_id)

PHASE3_WINNER = resolve_phase3_selected_condition()
STABILITY_CONDITIONS: list[ConditionSpec] = []
if PHASE3_WINNER is not None:
    STABILITY_CONDITIONS = [replace(
        PHASE3_WINNER,
        condition_id='FINALIST_STABILITY',
        phase='stability',
        description='Selected pre-LangChain finalist repeated across registered seeds.',
    )]
STABILITY_WORKFLOW = STABILITY_CONDITIONS[0].workflow if STABILITY_CONDITIONS else None
STABILITY_JOINT_TASKS = (
    make_task_schedule(BENCHMARK, STABILITY_CONDITIONS, STABILITY_SEEDS, 'stability', 'joint')
    if BENCHMARK is not None and STABILITY_CONDITIONS and STABILITY_WORKFLOW == 'W1_one_stage' and 'stability' in ACTIVE_PHASES
    else []
)
STABILITY_BINARY_TASKS = (
    make_task_schedule(BENCHMARK, STABILITY_CONDITIONS, STABILITY_SEEDS, 'stability', 'binary')
    if BENCHMARK is not None and STABILITY_CONDITIONS and STABILITY_WORKFLOW == 'W2_binary_first' and 'stability' in ACTIVE_PHASES
    else []
)
STABILITY_TARGET_TASKS = (
    make_task_schedule(BENCHMARK, STABILITY_CONDITIONS, STABILITY_SEEDS, 'stability', 'target')
    if BENCHMARK is not None and STABILITY_CONDITIONS and STABILITY_WORKFLOW == 'W2_binary_first' and 'stability' in ACTIVE_PHASES
    else []
)
display(pd.DataFrame([asdict(c) for c in STABILITY_CONDITIONS]))
print({
    'workflow':STABILITY_WORKFLOW,
    'seeds':STABILITY_SEEDS,
    'joint_tasks_per_provider':len(STABILITY_JOINT_TASKS),
    'binary_tasks_per_provider':len(STABILITY_BINARY_TASKS),
    'target_task_ceiling_per_provider':len(STABILITY_TARGET_TASKS),
})

### Stability Stage 1 / one-stage provider cells

In [ ]:
# STABILITY — OPENAI JOINT OR BINARY ONLY
STABILITY_OPENAI_STAGE1_RAW = pd.DataFrame()
STABILITY_OPENAI_STAGE1_TASKS = STABILITY_JOINT_TASKS or STABILITY_BINARY_TASKS
if STABILITY_OPENAI_STAGE1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    STABILITY_OPENAI_STAGE1_RAW = safe_run_provider_cell(
        'openai', STABILITY_OPENAI_STAGE1_TASKS, STABILITY_CONDITIONS
    )
    display(STABILITY_OPENAI_STAGE1_RAW.tail())
else:
    print('Stability OpenAI Stage 1/joint execution skipped.')

In [ ]:
# STABILITY — ANTHROPIC JOINT OR BINARY ONLY
STABILITY_ANTHROPIC_STAGE1_RAW = pd.DataFrame()
STABILITY_ANTHROPIC_STAGE1_TASKS = STABILITY_JOINT_TASKS or STABILITY_BINARY_TASKS
if STABILITY_ANTHROPIC_STAGE1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    STABILITY_ANTHROPIC_STAGE1_RAW = safe_run_provider_cell(
        'anthropic', STABILITY_ANTHROPIC_STAGE1_TASKS, STABILITY_CONDITIONS
    )
    display(STABILITY_ANTHROPIC_STAGE1_RAW.tail())
else:
    print('Stability Anthropic Stage 1/joint execution skipped.')

In [ ]:
# STABILITY — GEMINI JOINT OR BINARY ONLY
STABILITY_GEMINI_STAGE1_RAW = pd.DataFrame()
STABILITY_GEMINI_STAGE1_TASKS = STABILITY_JOINT_TASKS or STABILITY_BINARY_TASKS
if STABILITY_GEMINI_STAGE1_TASKS and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    STABILITY_GEMINI_STAGE1_RAW = safe_run_provider_cell(
        'gemini', STABILITY_GEMINI_STAGE1_TASKS, STABILITY_CONDITIONS
    )
    display(STABILITY_GEMINI_STAGE1_RAW.tail())
else:
    print('Stability Gemini Stage 1/joint execution skipped.')

In [ ]:
STABILITY_STAGE1_LOOKUPS = {
    provider: positive_stage1_lookup('stability', provider, 'FINALIST_STABILITY')
    for provider in MODEL_CONFIGS
} if STABILITY_WORKFLOW == 'W2_binary_first' else {provider:{} for provider in MODEL_CONFIGS}
print({provider:len(lookup) for provider,lookup in STABILITY_STAGE1_LOOKUPS.items()})

### Stability hierarchical Stage-2 provider cells

In [ ]:
# STABILITY W2 STAGE 2 — OPENAI ONLY
STABILITY_OPENAI_TARGET_RAW = pd.DataFrame()
if STABILITY_TARGET_TASKS and STABILITY_STAGE1_LOOKUPS.get('openai') and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    STABILITY_OPENAI_TARGET_RAW = safe_run_provider_cell(
        'openai', STABILITY_TARGET_TASKS, STABILITY_CONDITIONS,
        STABILITY_STAGE1_LOOKUPS['openai']
    )
    display(STABILITY_OPENAI_TARGET_RAW.tail())
else:
    print('Stability OpenAI Stage 2 skipped or not applicable.')

In [ ]:
# STABILITY W2 STAGE 2 — ANTHROPIC ONLY
STABILITY_ANTHROPIC_TARGET_RAW = pd.DataFrame()
if STABILITY_TARGET_TASKS and STABILITY_STAGE1_LOOKUPS.get('anthropic') and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    STABILITY_ANTHROPIC_TARGET_RAW = safe_run_provider_cell(
        'anthropic', STABILITY_TARGET_TASKS, STABILITY_CONDITIONS,
        STABILITY_STAGE1_LOOKUPS['anthropic']
    )
    display(STABILITY_ANTHROPIC_TARGET_RAW.tail())
else:
    print('Stability Anthropic Stage 2 skipped or not applicable.')

In [ ]:
# STABILITY W2 STAGE 2 — GEMINI ONLY
STABILITY_GEMINI_TARGET_RAW = pd.DataFrame()
if STABILITY_TARGET_TASKS and STABILITY_STAGE1_LOOKUPS.get('gemini') and (USE_MOCK_PROVIDER or (RUN_API_CALLS and API_RUN_CONFIRMATION == REQUIRED_API_CONFIRMATION)):
    STABILITY_GEMINI_TARGET_RAW = safe_run_provider_cell(
        'gemini', STABILITY_TARGET_TASKS, STABILITY_CONDITIONS,
        STABILITY_STAGE1_LOOKUPS['gemini']
    )
    display(STABILITY_GEMINI_TARGET_RAW.tail())
else:
    print('Stability Gemini Stage 2 skipped or not applicable.')

### Stability metrics, repeat agreement, and tiered review

In [ ]:
STABILITY_UNIFIED = assemble_workflow_predictions('stability')
STABILITY_SCORED = attach_gold_and_predictions(add_evidence_audit(STABILITY_UNIFIED)) if not STABILITY_UNIFIED.empty else pd.DataFrame()
STABILITY_RUN_METRICS = summarize_binary(
    STABILITY_SCORED,
    ['provider','seed','repeat_id','condition_id','definition_id','prompt_style','context_id','workflow']
) if not STABILITY_SCORED.empty else pd.DataFrame()
STABILITY_METRIC_DISTRIBUTION = pd.DataFrame()
if not STABILITY_RUN_METRICS.empty:
    metric_columns = [
        'accuracy','balanced_accuracy','precision_yes','recall_yes','specificity',
        'f1_yes','mcc','cohen_kappa','brier_score','schema_valid_rate',
        'evidence_valid_rate','estimated_cost_usd','mean_latency_seconds'
    ]
    rows = []
    for provider, group in STABILITY_RUN_METRICS.groupby('provider'):
        for metric in metric_columns:
            values = pd.to_numeric(group[metric], errors='coerce').dropna()
            rows.append({
                'provider':provider,'metric':metric,'n_runs':len(values),
                'mean':values.mean(),'std':values.std(ddof=1) if len(values)>1 else 0.0,
                'min':values.min() if len(values) else np.nan,
                'max':values.max() if len(values) else np.nan,
            })
    STABILITY_METRIC_DISTRIBUTION = pd.DataFrame(rows)
    display(STABILITY_RUN_METRICS)
    display(STABILITY_METRIC_DISTRIBUTION)
else:
    print('No stability predictions available yet.')

In [ ]:
def provider_repeat_stability(data: pd.DataFrame) -> tuple[pd.DataFrame,pd.DataFrame]:
    if data.empty:
        return pd.DataFrame(), pd.DataFrame()
    rows = []
    for (provider,row_id), group in data.groupby(['provider','row_id']):
        values = group.sort_values('seed')['pred_int'].dropna().astype(int).tolist()
        counts = Counter(values)
        modal_count = max(counts.values()) if counts else 0
        rows.append({
            'provider':provider,'row_id':row_id,'n_repeats':len(values),
            'yes_repeats':counts.get(1,0),'no_repeats':counts.get(0,0),
            'modal_prediction':max(counts, key=counts.get) if counts else np.nan,
            'repeat_agreement_share':safe_rate(modal_count,len(values)),
            'changed_across_repeats':len(counts)>1,
        })
    row_stability = pd.DataFrame(rows)
    pairwise = []
    for provider, provider_data in data.groupby('provider'):
        pivot = provider_data.pivot_table(index='row_id', columns='seed', values='pred_int', aggfunc='first')
        for left_seed,right_seed in itertools.combinations(pivot.columns,2):
            paired = pivot[[left_seed,right_seed]].dropna()
            if paired.empty:
                continue
            pairwise.append({
                'provider':provider,'seed_left':left_seed,'seed_right':right_seed,
                'n_rows':len(paired),
                'percent_agreement':paired[left_seed].eq(paired[right_seed]).mean(),
                'cohen_kappa':cohen_kappa_score(paired[left_seed],paired[right_seed])
                if paired[left_seed].nunique()>1 or paired[right_seed].nunique()>1 else np.nan,
            })
    return row_stability, pd.DataFrame(pairwise)

PROVIDER_ROW_STABILITY, PROVIDER_SEED_PAIRWISE = provider_repeat_stability(STABILITY_SCORED)
display(PROVIDER_ROW_STABILITY.head(30))
display(PROVIDER_SEED_PAIRWISE)

In [ ]:
def tier_from_yes_votes(yes_votes: int, n_votes: int) -> str:
    if n_votes != 3:
        return 'INCOMPLETE'
    return {3:'Tier 1',2:'Tier 2',1:'Tier 3',0:'Tier 4'}[int(yes_votes)]

def build_tiered_review(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    rows = []
    for (row_id,seed), group in data.groupby(['row_id','seed']):
        valid = group.dropna(subset=['pred_int'])
        yes_votes = int(valid['pred_int'].sum())
        n_votes = valid['provider'].nunique()
        tier = tier_from_yes_votes(yes_votes,n_votes)
        rows.append({
            'row_id':row_id,'seed':seed,'n_provider_votes':n_votes,
            'yes_votes':yes_votes,'no_votes':n_votes-yes_votes,
            'tier':tier,
            'majority_prediction':'Yes' if yes_votes>=2 else 'No',
            'providers_yes':' | '.join(sorted(valid.loc[valid['pred_int'].eq(1),'provider'].astype(str))),
            'providers_no':' | '.join(sorted(valid.loc[valid['pred_int'].eq(0),'provider'].astype(str))),
            'mean_confidence':valid['confidence'].mean(),
            'any_evidence_invalid':not bool(valid['evidence_valid'].fillna(False).all()),
        })
    return pd.DataFrame(rows).merge(
        BENCHMARK[['row_id','document_title','target_text','gold_historical','gold_final_definition']],
        on='row_id', how='left', validate='many_to_one'
    )

TIERED_REVIEW_BY_SEED = build_tiered_review(STABILITY_SCORED)
TIER_STABILITY = pd.DataFrame()
if not TIERED_REVIEW_BY_SEED.empty:
    tier_order = {'Tier 1':1,'Tier 2':2,'Tier 3':3,'Tier 4':4,'INCOMPLETE':9}
    rows=[]
    for row_id, group in TIERED_REVIEW_BY_SEED.groupby('row_id'):
        tiers=group.sort_values('seed')['tier'].tolist(); counts=Counter(tiers)
        rows.append({
            'row_id':row_id,'n_repeats':len(tiers),'tiers_by_seed':' | '.join(tiers),
            'distinct_tiers':len(counts),'modal_tier':counts.most_common(1)[0][0],
            'tier_stable':len(counts)==1,
            'tier_range':max(tier_order.get(t,9) for t in tiers)-min(tier_order.get(t,9) for t in tiers),
        })
    TIER_STABILITY = pd.DataFrame(rows).merge(
        BENCHMARK[['row_id','document_title','target_text']], on='row_id', how='left'
    )
    display(TIERED_REVIEW_BY_SEED.head(30))
    display(TIER_STABILITY.sort_values(['tier_stable','tier_range'], ascending=[True,False]))

In [ ]:
def fleiss_kappa_binary(vote_matrix: np.ndarray) -> float:
    matrix = np.asarray(vote_matrix, dtype=float)
    n_items, n_categories = matrix.shape
    n_raters = matrix.sum(axis=1)
    if n_items == 0 or np.any(n_raters < 2) or not np.allclose(n_raters,n_raters[0]):
        return np.nan
    n = n_raters[0]
    p = matrix.sum(axis=0) / (n_items*n)
    p_bar_e = np.sum(p**2)
    p_i = (np.sum(matrix**2,axis=1)-n)/(n*(n-1))
    p_bar = p_i.mean()
    return float((p_bar-p_bar_e)/(1-p_bar_e)) if p_bar_e < 1 else np.nan

def cross_provider_reliability(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame()
    rows=[]
    try:
        import krippendorff as kd
    except ImportError:
        kd=None
    for seed, group in data.groupby('seed'):
        pivot=group.pivot_table(index='row_id',columns='provider',values='pred_int',aggfunc='first')
        complete=pivot.dropna()
        if complete.empty:
            continue
        counts=np.column_stack([(complete==0).sum(axis=1),(complete==1).sum(axis=1)])
        alpha=np.nan
        if kd is not None:
            with contextlib.suppress(Exception):
                alpha=float(kd.alpha(reliability_data=complete.to_numpy().T,level_of_measurement='nominal'))
        rows.append({
            'seed':seed,'rows_complete':len(complete),'providers':len(complete.columns),
            'unanimous_agreement':complete.nunique(axis=1).eq(1).mean(),
            'fleiss_kappa':fleiss_kappa_binary(counts),
            'krippendorff_alpha':alpha,
        })
    return pd.DataFrame(rows)

CROSS_PROVIDER_RELIABILITY = cross_provider_reliability(STABILITY_SCORED)
display(CROSS_PROVIDER_RELIABILITY)

### Document-stratified bootstrap uncertainty

In [ ]:
def document_stratified_bootstrap(
    data: pd.DataFrame,
    group_columns: Sequence[str] = ('provider',),
    metrics: Sequence[str] = ('accuracy','recall_yes','specificity','f1_yes','mcc'),
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = 20260722,
) -> pd.DataFrame:
    if data.empty or n_bootstrap <= 0:
        return pd.DataFrame()
    rng=np.random.default_rng(seed)
    results=[]
    grouper=group_columns[0] if len(group_columns)==1 else list(group_columns)
    for keys,group in data.groupby(grouper,dropna=False):
        keys=keys if isinstance(keys,tuple) else (keys,)
        document_groups={doc:doc_frame for doc,doc_frame in group.groupby('document_title')}
        estimates={metric:[] for metric in metrics}
        for _ in range(n_bootstrap):
            pieces=[]
            for document,doc_frame in document_groups.items():
                indices=rng.integers(0,len(doc_frame),size=len(doc_frame))
                pieces.append(doc_frame.iloc[indices])
            sample=pd.concat(pieces,ignore_index=True)
            record=binary_metric_record(sample,'gold_common_int')
            for metric in metrics:
                estimates[metric].append(record.get(metric,np.nan))
        point=binary_metric_record(group,'gold_common_int')
        for metric,values in estimates.items():
            values=np.asarray(values,dtype=float); values=values[~np.isnan(values)]
            results.append({
                **dict(zip(group_columns,keys)),'metric':metric,
                'point_estimate':point.get(metric,np.nan),'n_bootstrap':len(values),
                'ci_2_5':np.quantile(values,0.025) if len(values) else np.nan,
                'ci_97_5':np.quantile(values,0.975) if len(values) else np.nan,
            })
    return pd.DataFrame(results)

STABILITY_BOOTSTRAP_CI = document_stratified_bootstrap(STABILITY_SCORED)
display(STABILITY_BOOTSTRAP_CI)

In [ ]:
def final_review_queue() -> pd.DataFrame:
    if TIERED_REVIEW_BY_SEED.empty:
        return pd.DataFrame()
    tier_summary=TIER_STABILITY.copy()
    per_row=(
        TIERED_REVIEW_BY_SEED.groupby('row_id')
        .agg(
            seeds=('seed','nunique'),
            any_tier1=('tier',lambda x:any(v=='Tier 1' for v in x)),
            any_tier2=('tier',lambda x:any(v=='Tier 2' for v in x)),
            any_tier3=('tier',lambda x:any(v=='Tier 3' for v in x)),
            any_incomplete=('tier',lambda x:any(v=='INCOMPLETE' for v in x)),
            evidence_invalid_any=('any_evidence_invalid','max'),
            mean_confidence=('mean_confidence','mean'),
        ).reset_index()
    )
    provider_instability=(
        PROVIDER_ROW_STABILITY.groupby('row_id')
        .agg(
            unstable_providers=('changed_across_repeats','sum'),
            minimum_repeat_agreement=('repeat_agreement_share','min'),
        ).reset_index()
        if not PROVIDER_ROW_STABILITY.empty else pd.DataFrame({'row_id':BENCHMARK['row_id']})
    )
    queue=tier_summary.merge(per_row,on='row_id',how='left').merge(provider_instability,on='row_id',how='left')
    queue['review_priority_score']=(
        (~queue['tier_stable']).astype(int)*4
        + queue['tier_range'].fillna(0).clip(upper=4)
        + queue['any_tier2'].astype(int)*2
        + queue['any_tier3'].astype(int)*3
        + queue['evidence_invalid_any'].fillna(False).astype(int)*4
        + queue['unstable_providers'].fillna(0)
    )
    return queue.sort_values(
        ['review_priority_score','tier_range','minimum_repeat_agreement'],
        ascending=[False,False,True]
    )

FINAL_REVIEW_QUEUE = final_review_queue()
display(FINAL_REVIEW_QUEUE.head(50))

## 22. Definition-impact diagnostics with special attention to EPA

In [ ]:
def definition_impact_table(phase1_scored: pd.DataFrame) -> pd.DataFrame:
    if phase1_scored.empty:
        return pd.DataFrame()
    subset=phase1_scored[
        phase1_scored['definition_id'].isin(['D1_old_broad','D2_current_strict','D3_provisional_adaptive'])
    ].copy()
    rows=[]
    for (provider,prompt_style,document),group in subset.groupby(['provider','prompt_style','document_title']):
        pivot=group.pivot_table(
            index=['row_id','seed'],columns='definition_id',values='pred_int',aggfunc='first'
        )
        for left,right in [('D1_old_broad','D2_current_strict'),('D2_current_strict','D3_provisional_adaptive'),('D1_old_broad','D3_provisional_adaptive')]:
            if left not in pivot or right not in pivot:
                continue
            paired=pivot[[left,right]].dropna()
            rows.append({
                'provider':provider,'prompt_style':prompt_style,'document_title':document,
                'contrast':f'{left} -> {right}','n_paired':len(paired),
                'no_to_yes':int(((paired[left]==0)&(paired[right]==1)).sum()),
                'yes_to_no':int(((paired[left]==1)&(paired[right]==0)).sum()),
                'unchanged_yes':int(((paired[left]==1)&(paired[right]==1)).sum()),
                'unchanged_no':int(((paired[left]==0)&(paired[right]==0)).sum()),
            })
    return pd.DataFrame(rows)

DEFINITION_IMPACT = definition_impact_table(PHASE1_SCORED)
D3_BOUNDARY_CASES = pd.DataFrame()
if not PHASE1_SCORED.empty:
    d2=PHASE1_SCORED[PHASE1_SCORED['definition_id'].eq('D2_current_strict')][
        ['provider','prompt_style','row_id','seed','pred_int','out__change_type','out__unlearning_mode']
    ].rename(columns={
        'pred_int':'d2_pred','out__change_type':'d2_change_type','out__unlearning_mode':'d2_mode'
    })
    d3=PHASE1_SCORED[PHASE1_SCORED['definition_id'].eq('D3_provisional_adaptive')][
        ['provider','prompt_style','row_id','seed','pred_int','out__change_type','out__unlearning_mode','evidence_valid']
    ].rename(columns={
        'pred_int':'d3_pred','out__change_type':'d3_change_type','out__unlearning_mode':'d3_mode'
    })
    D3_BOUNDARY_CASES=d2.merge(d3,on=['provider','prompt_style','row_id','seed'],how='inner')
    D3_BOUNDARY_CASES=D3_BOUNDARY_CASES[
        D3_BOUNDARY_CASES['d2_pred'].ne(D3_BOUNDARY_CASES['d3_pred'])
    ].merge(
        BENCHMARK[['row_id','document_title','target_text','gold_historical']],
        on='row_id',how='left'
    )
    D3_BOUNDARY_CASES['is_epa']=D3_BOUNDARY_CASES['document_title'].str.contains('EPA',case=False,na=False)
    display(DEFINITION_IMPACT)
    display(D3_BOUNDARY_CASES.sort_values(['is_epa','row_id'],ascending=[False,True]))

**Interpretation rule:** D3 is promising only when it converts theoretically plausible durable reconfiguration cases while preserving non-EPA specificity and rejecting capacity-only additions. Higher agreement with legacy labels is not itself evidence that D3 is the correct construct.

## 23. Run completeness, manifests, and final configuration

In [ ]:
def run_completeness_table() -> pd.DataFrame:
    expected_rows=[]
    phase_specs={
        'phase1':PHASE1_TASKS,
        'phase2':PHASE2_TASKS,
        'phase3':PHASE3_W1_TASKS+PHASE3_W2_BINARY_TASKS+PHASE3_W2_TARGET_TASKS,
        'stability':STABILITY_JOINT_TASKS+STABILITY_BINARY_TASKS+STABILITY_TARGET_TASKS,
    }
    for phase,tasks in phase_specs.items():
        for provider in MODEL_CONFIGS:
            latest=provider_results_snapshot(phase_log_paths(phase,provider)['results'])
            successful=latest[latest.get('status',pd.Series(dtype=str)).eq('ok')] if not latest.empty else pd.DataFrame()
            for stage in sorted(set(task.stage for task in tasks) or {'none'}):
                expected=sum(task.stage==stage for task in tasks)
                observed=int(successful.get('stage',pd.Series(dtype=str)).eq(stage).sum()) if not successful.empty else 0
                # Target-stage expected is a ceiling because negative Stage 1 rows are intentionally skipped.
                expected_kind='ceiling' if stage=='target' else 'exact'
                expected_rows.append({
                    'phase':phase,'provider':provider,'stage':stage,
                    'expected_tasks':expected,'expected_kind':expected_kind,
                    'successful_records':observed,
                    'complete':observed==expected if expected_kind=='exact' else observed<=expected,
                })
    return pd.DataFrame(expected_rows)

RUN_COMPLETENESS = run_completeness_table()
display(RUN_COMPLETENESS)

In [ ]:
FINAL_CONDITION = STABILITY_CONDITIONS[0] if STABILITY_CONDITIONS else PHASE3_SELECTED_CONDITION
FINAL_CONFIGURATION = {
    'protocol_version':PROTOCOL_VERSION,
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'dataset_sha256':DATASET_SHA256,
    'benchmark_rows':None if BENCHMARK is None else len(BENCHMARK),
    'gold_common_final_basis':common_final_gold_column()[1],
    'definition_is_provisional':bool(FINAL_CONDITION and FINAL_CONDITION.definition_id=='D3_provisional_adaptive'),
    'condition':None if FINAL_CONDITION is None else asdict(FINAL_CONDITION),
    'models':{provider:asdict(config) for provider,config in MODEL_CONFIGS.items()},
    'temperature':0.0,
    'stability_seeds':STABILITY_SEEDS,
    'one_target_paragraph_per_request':True,
    'few_shot_examples_used':False,
    'structured_evidence_only':True,
    'tiered_review_policy':{
        'Tier 1':'3 of 3 providers Yes',
        'Tier 2':'2 of 3 providers Yes',
        'Tier 3':'1 of 3 providers Yes',
        'Tier 4':'0 of 3 providers Yes',
    },
    'selection_constraints':{
        'minimum_schema_valid_rate':MIN_SCHEMA_VALID_RATE,
        'minimum_evidence_valid_rate':MIN_EVIDENCE_VALID_RATE,
        'specificity_floor':SPECIFICITY_FLOOR,
    },
    'warning':'Do not treat D3 as final until human adjudication is completed.',
}
atomic_write_text(
    CONFIG_ROOT / 'final_configuration.json',
    json.dumps(FINAL_CONFIGURATION,indent=2,default=str)
)
print(json.dumps(FINAL_CONFIGURATION,indent=2,default=str))

## 24. Export reproducible tables and a formatted analysis workbook

In [ ]:
def exportable_tables() -> dict[str,pd.DataFrame]:
    candidates={
        'Config Summary':pd.DataFrame([CONFIG_SUMMARY]),
        'Models':MODEL_PROTOCOL,
        'Definitions':DEFINITION_REGISTRY,
        'Gold Availability':GOLD_AVAILABILITY,
        'Benchmark Audit':BENCHMARK_AUDIT,
        'Prompt Audit':PROMPT_AUDIT,
        'Phase1 Ranking':PHASE1_RANKING,
        'Phase1 Provider':PHASE1_PROVIDER_METRICS,
        'Phase1 Documents':PHASE1_DOCUMENT_METRICS,
        'Phase1 Aligned':PHASE1_ALIGNED_METRICS,
        'Phase1 Tradeoff':PHASE1_DEFINITION_TRADEOFF,
        'Phase1 EPA':PHASE1_EPA_DIAGNOSTICS,
        'Phase1 McNemar':PHASE1_MCNEMAR,
        'Phase1 GEE':PHASE1_GEE,
        'Definition Impact':DEFINITION_IMPACT,
        'D3 Boundary Cases':D3_BOUNDARY_CASES,
        'Phase2 Ranking':PHASE2_RANKING,
        'Phase2 Provider':PHASE2_PROVIDER_METRICS,
        'Phase2 Documents':PHASE2_DOCUMENT_METRICS,
        'Phase2 McNemar':PHASE2_MCNEMAR,
        'Phase2 GEE':PHASE2_GEE,
        'Phase3 Ranking':PHASE3_RANKING,
        'Phase3 Provider':PHASE3_PROVIDER_METRICS,
        'Phase3 Documents':PHASE3_DOCUMENT_METRICS,
        'Phase3 Targets':PHASE3_TARGET_METRICS,
        'Phase3 McNemar':PHASE3_MCNEMAR,
        'Phase3 GEE':PHASE3_GEE,
        'Stability Runs':STABILITY_RUN_METRICS,
        'Stability Distribution':STABILITY_METRIC_DISTRIBUTION,
        'Provider Row Stability':PROVIDER_ROW_STABILITY,
        'Seed Pairwise':PROVIDER_SEED_PAIRWISE,
        'Tiered Review':TIERED_REVIEW_BY_SEED,
        'Tier Stability':TIER_STABILITY,
        'Reliability':CROSS_PROVIDER_RELIABILITY,
        'Bootstrap CI':STABILITY_BOOTSTRAP_CI,
        'Review Queue':FINAL_REVIEW_QUEUE,
        'Completeness':RUN_COMPLETENESS,
    }
    return {name:frame for name,frame in candidates.items() if isinstance(frame,pd.DataFrame) and not frame.empty}

EXPORT_TABLES=exportable_tables()
TABLES_DIR=REPORTS_ROOT/'tables'
TABLES_DIR.mkdir(parents=True,exist_ok=True)
TABLE_MANIFEST=[]
for name,frame in EXPORT_TABLES.items():
    filename=re.sub(r'[^A-Za-z0-9_-]+','_',name).strip('_').lower()+'.csv'
    path=TABLES_DIR/filename
    frame.to_csv(path,index=False)
    TABLE_MANIFEST.append({
        'table':name,'filename':filename,'rows':len(frame),'columns':len(frame.columns),
        'sha256':sha256_file(path),
    })
TABLE_MANIFEST_DF=pd.DataFrame(TABLE_MANIFEST)
TABLE_MANIFEST_DF.to_csv(REPORTS_ROOT/'table_manifest.csv',index=False)
display(TABLE_MANIFEST_DF)

In [ ]:
def unique_sheet_name(name: str, used: set[str]) -> str:
    base=re.sub(r'[\\/*?:\[\]]','_',name)[:31] or 'Sheet'
    candidate=base; counter=1
    while candidate in used:
        suffix=f'_{counter}'
        candidate=base[:31-len(suffix)]+suffix
        counter+=1
    used.add(candidate)
    return candidate

def format_excel_worksheet(worksheet) -> None:
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    worksheet.freeze_panes='A2'
    worksheet.auto_filter.ref=worksheet.dimensions
    header_fill=PatternFill('solid',fgColor='1F4E78')
    header_font=Font(color='FFFFFF',bold=True)
    for cell in worksheet[1]:
        cell.fill=header_fill; cell.font=header_font
        cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
    for column_cells in worksheet.columns:
        letter=get_column_letter(column_cells[0].column)
        max_length=max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells[:200])
        worksheet.column_dimensions[letter].width=min(max(max_length+2,10),60)
    for row in worksheet.iter_rows(min_row=2):
        for cell in row:
            cell.alignment=Alignment(vertical='top',wrap_text=True)

REPORT_WORKBOOK=REPORTS_ROOT/'prelangchain_ab_results.xlsx'
if EXPORT_TABLES:
    with pd.ExcelWriter(REPORT_WORKBOOK,engine='openpyxl') as writer:
        used=set()
        for name,frame in EXPORT_TABLES.items():
            sheet=unique_sheet_name(name,used)
            frame.to_excel(writer,sheet_name=sheet,index=False)
        for worksheet in writer.book.worksheets:
            format_excel_worksheet(worksheet)
    print('Saved:',REPORT_WORKBOOK)
else:
    print('No result tables available; workbook not written.')

In [ ]:
def artifact_manifest(root: Path) -> pd.DataFrame:
    rows=[]
    for path in sorted(root.rglob('*')):
        if path.is_file():
            rows.append({
                'relative_path':str(path.relative_to(root)),
                'size_bytes':path.stat().st_size,
                'modified_utc':datetime.fromtimestamp(path.stat().st_mtime,tz=timezone.utc).isoformat(),
                'sha256':sha256_file(path),
            })
    return pd.DataFrame(rows)

ARTIFACT_MANIFEST=artifact_manifest(PROJECT_ROOT)
ARTIFACT_MANIFEST.to_csv(REPORTS_ROOT/'artifact_manifest.csv',index=False)
print('Manifested files:',len(ARTIFACT_MANIFEST))

## 25. Protocol self-tests

In [ ]:
def protocol_self_tests() -> pd.DataFrame:
    checks=[]
    def add(name: str, passed: bool, observed: Any, requirement: str, severity: str='error'):
        checks.append({'check':name,'passed':bool(passed),'observed':observed,'requirement':requirement,'severity':severity})
    add('all model temperatures are zero',all(c.temperature==0.0 for c in MODEL_CONFIGS.values()),[c.temperature for c in MODEL_CONFIGS.values()],'all 0.0')
    add('three distinct providers',set(MODEL_CONFIGS)=={'openai','anthropic','gemini'},sorted(MODEL_CONFIGS),'openai, anthropic, gemini')
    add('no rationale schema fields',not SCHEMA_AUDIT['has_rationale_field'].any(),SCHEMA_AUDIT['has_rationale_field'].sum(),'0')
    add('no few-shot condition',not any('example' in c.condition_id.casefold() or 'example' in c.description.casefold() for c in PHASE1_CONDITIONS),[c.condition_id for c in PHASE1_CONDITIONS],'none')
    add('phase1 condition IDs unique',PHASE1_REGISTRY['condition_id'].is_unique,PHASE1_REGISTRY['condition_id'].nunique(),len(PHASE1_REGISTRY))
    add('target appears once in audited prompts',PROMPT_AUDIT.empty or PROMPT_AUDIT['target_occurrences_normalized'].eq(1).all(),None if PROMPT_AUDIT.empty else PROMPT_AUDIT['target_occurrences_normalized'].tolist(),'all 1')
    add('no gold leakage in audited prompts',PROMPT_AUDIT.empty or not PROMPT_AUDIT['contains_forbidden_term'].any(),None if PROMPT_AUDIT.empty else PROMPT_AUDIT['contains_forbidden_term'].sum(),'0')
    add('dataset hash recorded',bool(DATASET_SHA256),DATASET_SHA256,'nonempty SHA-256')
    add('definition hashes unique',DEFINITION_REGISTRY['sha256'].nunique()==len(DEFINITION_REGISTRY),DEFINITION_REGISTRY['sha256'].nunique(),len(DEFINITION_REGISTRY))
    add('provider directories isolated',len({str(provider_phase_directory('phase1',p)) for p in MODEL_CONFIGS})==3,[str(provider_phase_directory('phase1',p)) for p in MODEL_CONFIGS],'3 unique paths')
    if USE_MOCK_PROVIDER and not PHASE1_SCORED.empty:
        duplicates=PHASE1_SCORED.groupby(['provider','condition_id','row_id','seed'])['pred_int'].nunique().max()
        add('mock deterministic within run key',duplicates<=1,duplicates,'<=1 unique prediction',severity='error')
    if not STABILITY_SCORED.empty:
        add('registered stability seeds observed',set(STABILITY_SCORED['seed'])==set(STABILITY_SEEDS),sorted(STABILITY_SCORED['seed'].unique()),sorted(STABILITY_SEEDS))
        add('all three providers in stability',set(STABILITY_SCORED['provider'])==set(MODEL_CONFIGS),sorted(STABILITY_SCORED['provider'].unique()),sorted(MODEL_CONFIGS))
    return pd.DataFrame(checks)

PROTOCOL_SELF_TESTS=protocol_self_tests()
display(PROTOCOL_SELF_TESTS)
failed_errors=PROTOCOL_SELF_TESTS[
    ~PROTOCOL_SELF_TESTS['passed'] & PROTOCOL_SELF_TESTS['severity'].eq('error')
]
if not failed_errors.empty:
    raise AssertionError('Protocol self-tests failed:\n'+failed_errors.to_string(index=False))

In [ ]:
EXECUTION_SUMMARY={
    'protocol_version':PROTOCOL_VERSION,
    'completed_at_utc':datetime.now(timezone.utc).isoformat(),
    'synthetic_fixture':USE_SYNTHETIC_DATA,
    'mock_provider':USE_MOCK_PROVIDER,
    'paid_api_calls_enabled':RUN_API_CALLS,
    'dataset_sha256':DATASET_SHA256,
    'phase1_unified_predictions':len(PHASE1_SCORED),
    'phase2_unified_predictions':len(PHASE2_SCORED),
    'phase3_unified_predictions':len(PHASE3_SCORED),
    'stability_unified_predictions':len(STABILITY_SCORED),
    'phase1_selection':PHASE1_SELECTION,
    'phase2_selection':PHASE2_SELECTION,
    'phase3_selection':PHASE3_SELECTION,
    'final_configuration_path':str(CONFIG_ROOT/'final_configuration.json'),
    'report_workbook':str(REPORT_WORKBOOK) if REPORT_WORKBOOK.exists() else None,
    'protocol_self_tests_passed':bool(PROTOCOL_SELF_TESTS['passed'].all()),
}
atomic_write_text(
    REPORTS_ROOT/'execution_summary.json',
    json.dumps(EXECUTION_SUMMARY,indent=2,default=str)
)
print(json.dumps(EXECUTION_SUMMARY,indent=2,default=str))

## 26. How to run this protocol on the real benchmark

1. Place `Unlearning_Codebook_Local_Context_Test_Set.xlsx` at `data/` under the project root, or set `UNLEARNING_INPUT_WORKBOOK`.
2. Run the notebook once with APIs disabled. Inspect benchmark, prompt, schema, and model registries.
3. Add API keys to environment variables—not notebook cells.
4. Run the three smoke-test cells independently.
5. Set:

```bash
export UNLEARNING_RUN_API_CALLS=true
export UNLEARNING_API_CONFIRMATION=RUN_PRELANGCHAIN_AB_V1
export UNLEARNING_ACTIVE_PHASES=phase1
export UNLEARNING_SEEDS=17
```

6. Run Phase 1 provider cells separately. Inspect errors and selection before enabling Phase 2.
7. Repeat with `UNLEARNING_ACTIVE_PHASES=phase2`, then `phase3`, then `stability`.
8. Rerunning a provider cell resumes successful deterministic run keys; it does not overwrite raw logs.
9. For an independent notebook-level replication, change `UNLEARNING_REPLICATE_ID` while retaining the same registered seeds and frozen inputs.
10. Human adjudication should later add complete `Gold Old Definition`, `Gold Current Definition`, and `Gold Final Definition` columns. Rerun analysis from the preserved raw outputs where the prompts remain applicable; run new calls only when the definition treatment itself changes.

### Recommended independent replication IDs

```text
r1_discovery
r2_clean_kernel
r3_independent_day
```

Do not pool provider votes and repeated runs as if all observations were independent. Tiered review is calculated within each seed; repeat stability is reported separately.

## 27. Deferred LangChain phase

This notebook intentionally contains no retrieved or within-sheet examples. The later LangChain experiment should reuse the frozen benchmark, definition registry, provider adapters, evidence schemas, logging, evaluation, and tiered-review functions. Only example retrieval and prompt assembly should be added.

For each target, exclude the target row, exact/near-duplicate cluster, evaluation rows, and preferably the entire source document from the example pool. That later experiment becomes a new protocol version rather than an edit to this one.